In [ ]:
!pip install ultralytics

In [ ]:
# ================================================================
# CROWD DETECTION PROJECT — V1 ML PHASE
# CELL 1: VERIFY TRAINED YOLO26m MODEL
# ================================================================

from pathlib import Path
import os
import torch

print("=" * 75)
print(" CROWD DETECTION PROJECT — V1 ML PHASE")
print(" CELL 1: TRAINED YOLO26m MODEL VERIFICATION")
print("=" * 75)

# ----------------------------------------------------------------
# STEP 1 — Check Python / PyTorch environment
# ----------------------------------------------------------------

print("\n[1/5] Checking environment...")
print("-" * 75)

print(f"Python executable : {os.sys.executable}")
print(f"PyTorch version   : {torch.__version__}")
print(f"CUDA available    : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU               : {torch.cuda.get_device_name(0)}")
    print(f"CUDA version      : {torch.version.cuda}")
else:
    print("GPU               : Not available")
    print("⚠️  YOLO will run on CPU.")

# ----------------------------------------------------------------
# STEP 2 — Locate the model
# ----------------------------------------------------------------

print("\n[2/5] Looking for trained YOLO model...")
print("-" * 75)

# Possible locations.
possible_models = [
    "/content/best.pt",
    "/content/yolo26m_crowdhuman_best.pt",
    "/content/runs/crowdhuman/yolo26m_v1/weights/best.pt",
]

model_path = None

for path in possible_models:
    if Path(path).exists():
        model_path = Path(path)
        print(f"✅ Found model:")
        print(f"   {model_path}")
        break

if model_path is None:
    print("❌ Could not automatically find best.pt")
    print("\nFiles currently available in /content:")

    for p in Path("/content").iterdir():
        print(f"   {p}")

    raise FileNotFoundError(
        "\nPlease upload your trained best.pt to Colab "
        "and run this cell again."
    )

# ----------------------------------------------------------------
# STEP 3 — Check model file
# ----------------------------------------------------------------

print("\n[3/5] Checking model file...")
print("-" * 75)

model_size_mb = model_path.stat().st_size / (1024 ** 2)

print(f"Model path : {model_path}")
print(f"Model size : {model_size_mb:.2f} MB")

if model_size_mb < 50:
    print("⚠️  Model is unusually small. Please verify this is the trained model.")
else:
    print("✅ Model size looks reasonable.")

# ----------------------------------------------------------------
# STEP 4 — Load YOLO
# ----------------------------------------------------------------

print("\n[4/5] Loading trained YOLO26m model...")
print("-" * 75)

from ultralytics import YOLO

print("Importing Ultralytics...")
print("Loading model...")
print("⏳ Please wait...\n")

model = YOLO(str(model_path))

print("✅ Model loaded successfully!")

# ----------------------------------------------------------------
# STEP 5 — Inspect model
# ----------------------------------------------------------------

print("\n[5/5] Inspecting model...")
print("-" * 75)

print(f"Model path       : {model_path}")
print(f"Model task       : {model.task}")

print("\nModel class names:")

if hasattr(model, "names"):
    for class_id, class_name in model.names.items():
        print(f"   {class_id} → {class_name}")

print("\nModel information:")
model.info()

print("\n" + "=" * 75)
print(" MODEL VERIFICATION COMPLETE")
print("=" * 75)

print("\n✅ If you see:")
print("   • best.pt found")
print("   • model loaded successfully")
print("   • class 0 → person")
print("   • YOLO26m model information")
print("\nthen our trained model is ready for the ML pipeline.")
print("=" * 75)

In [ ]:
# ============================================================
# CELL 1 — DOWNLOAD CROWDHUMAN FROM HUGGING FACE
# ============================================================

# Install the Hugging Face CLI
!pip install -q -U huggingface_hub

from huggingface_hub import snapshot_download
from pathlib import Path

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

REPO_ID = "sshao0516/CrowdHuman"
LOCAL_DIR = "/content/CrowdHuman"

print("=" * 60)
print("CrowdHuman Dataset Downloader")
print("=" * 60)

print(f"\nRepository : {REPO_ID}")
print(f"Download to: {LOCAL_DIR}")

# ------------------------------------------------------------
# Download the complete original dataset
# ------------------------------------------------------------

print("\nStarting download...")
print("⚠️ The complete dataset is approximately 14.2 GB.")
print("This may take some time depending on Colab's connection.\n")

snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    local_dir=LOCAL_DIR
)

print("\n" + "=" * 60)
print("DOWNLOAD COMPLETE ✅")
print("=" * 60)

# ------------------------------------------------------------
# Show downloaded files
# ------------------------------------------------------------

print("\nFiles downloaded:")

for file in sorted(Path(LOCAL_DIR).iterdir()):
    if file.is_file():
        size_gb = file.stat().st_size / (1024 ** 3)
        print(f"  {file.name:<30} {size_gb:.2f} GB")

In [ ]:
# ============================================================
# CELL 2 — VERIFY CROWDHUMAN ANNOTATIONS
# ============================================================

import json
from pathlib import Path

DATASET = Path("/content/CrowdHuman")

TRAIN_ANN = DATASET / "annotation_train.odgt"
VAL_ANN   = DATASET / "annotation_val.odgt"

print("=" * 70)
print("CrowdHuman Dataset Verification")
print("=" * 70)

# ------------------------------------------------------------
# 1. Check required files
# ------------------------------------------------------------

required_files = [
    "CrowdHuman_train01.zip",
    "CrowdHuman_train02.zip",
    "CrowdHuman_train03.zip",
    "CrowdHuman_val.zip",
    "annotation_train.odgt",
    "annotation_val.odgt",
]

print("\n[1] Checking required files...\n")

all_present = True

for filename in required_files:
    path = DATASET / filename

    if path.exists():
        size_mb = path.stat().st_size / (1024 ** 2)
        print(f"✅ {filename:<30} {size_mb:>10.2f} MB")
    else:
        print(f"❌ {filename:<30} MISSING")
        all_present = False

# ------------------------------------------------------------
# 2. Read one training annotation
# ------------------------------------------------------------

print("\n[2] Inspecting training annotation format...\n")

with open(TRAIN_ANN, "r") as f:
    first_line = f.readline()

annotation = json.loads(first_line)

print("Image ID:")
print(" ", annotation.get("ID"))

print("\nImage size:")
print(" ", annotation.get("width"), "x", annotation.get("height"))

print("\nNumber of ground-truth boxes:")
print(" ", len(annotation.get("gtboxes", [])))

# ------------------------------------------------------------
# 3. Inspect first few ground-truth boxes
# ------------------------------------------------------------

print("\n[3] Inspecting first 5 ground-truth annotations...\n")

for i, gtbox in enumerate(annotation.get("gtboxes", [])[:5]):

    print(f"Person/Box #{i + 1}")

    print("  tag :", gtbox.get("tag"))
    print("  fbox:", gtbox.get("fbox"))
    print()

# ------------------------------------------------------------
# 4. Count annotations
# ------------------------------------------------------------

print("[4] Counting training and validation images...\n")

def count_lines(annotation_file):

    count = 0

    with open(annotation_file, "r") as f:
        for line in f:
            if line.strip():
                count += 1

    return count


train_count = count_lines(TRAIN_ANN)
val_count = count_lines(VAL_ANN)

print(f"Training annotation entries  : {train_count}")
print(f"Validation annotation entries: {val_count}")

# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print("\n" + "=" * 70)

if all_present:
    print("DATASET FILE CHECK: PASSED ✅")
else:
    print("DATASET FILE CHECK: FAILED ❌")

print("=" * 70)

In [ ]:
# ============================================================
# CELL 3 — EXTRACT CROWDHUMAN TRAIN + VALIDATION IMAGES
# ============================================================

import zipfile
from pathlib import Path
import shutil

DATASET = Path("/content/CrowdHuman")

# ------------------------------------------------------------
# Create extraction directories
# ------------------------------------------------------------

TRAIN_DIR = DATASET / "train_images"
VAL_DIR   = DATASET / "val_images"

TRAIN_DIR.mkdir(exist_ok=True)
VAL_DIR.mkdir(exist_ok=True)

print("=" * 70)
print("CrowdHuman Image Extraction")
print("=" * 70)

# ------------------------------------------------------------
# Training ZIP files
# ------------------------------------------------------------

train_zips = [
    DATASET / "CrowdHuman_train01.zip",
    DATASET / "CrowdHuman_train02.zip",
    DATASET / "CrowdHuman_train03.zip",
]

print("\n[1] Extracting TRAINING images")
print("-" * 70)

for zip_path in train_zips:

    print(f"\n📦 {zip_path.name}")

    with zipfile.ZipFile(zip_path, "r") as zip_ref:

        members = zip_ref.namelist()

        print(f"   Files inside ZIP: {len(members)}")
        print("   Extracting...")

        zip_ref.extractall(TRAIN_DIR)

    print("   ✅ Done")


# ------------------------------------------------------------
# Validation ZIP
# ------------------------------------------------------------

print("\n[2] Extracting VALIDATION images")
print("-" * 70)

print(f"\n📦 {DATASET / 'CrowdHuman_val.zip'}")

with zipfile.ZipFile(DATASET / "CrowdHuman_val.zip", "r") as zip_ref:

    members = zip_ref.namelist()

    print(f"   Files inside ZIP: {len(members)}")
    print("   Extracting...")

    zip_ref.extractall(VAL_DIR)

print("   ✅ Done")


# ------------------------------------------------------------
# Count extracted image files
# ------------------------------------------------------------

def count_images(directory):

    extensions = {".jpg", ".jpeg", ".png"}

    return [
        path for path in directory.rglob("*")
        if path.suffix.lower() in extensions
    ]


train_images = count_images(TRAIN_DIR)
val_images = count_images(VAL_DIR)


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("EXTRACTION SUMMARY")
print("=" * 70)

print(f"\nTraining images extracted   : {len(train_images)}")
print(f"Validation images extracted: {len(val_images)}")

print("\nExpected:")
print("Training images             : 15000")
print("Validation images           : 4370")

print("\n" + "=" * 70)

if len(train_images) == 15000:
    print("✅ Training extraction looks correct")
else:
    print("⚠️ Training image count differs from expected")

if len(val_images) == 4370:
    print("✅ Validation extraction looks correct")
else:
    print("⚠️ Validation image count differs from expected")

print("=" * 70)


# ============================================================
# CELL 4 — VERIFY IMAGE / ANNOTATION MATCHING
# ============================================================
#
# Goal:
#   Make sure every CrowdHuman annotation has a corresponding
#   image before we start converting anything to YOLO format.
#
# CrowdHuman annotation:
#   ID = "284193,faa9000f2678b5e"
#
# Image:
#   284193,faa9000f2678b5e.jpg
#
# If these match correctly, we are ready for YOLO conversion.
# ============================================================

from pathlib import Path
import json

DATASET = Path("/content/CrowdHuman")

TRAIN_IMAGE_DIR = DATASET / "train_images"
VAL_IMAGE_DIR   = DATASET / "val_images"

TRAIN_ANN = DATASET / "annotation_train.odgt"
VAL_ANN   = DATASET / "annotation_val.odgt"


# ------------------------------------------------------------
# Build an index of all extracted images
# ------------------------------------------------------------

def build_image_index(image_directory):

    image_index = {}

    for image_path in image_directory.rglob("*"):

        if image_path.suffix.lower() in [".jpg", ".jpeg", ".png"]:

            # Filename without extension
            image_id = image_path.stem

            image_index[image_id] = image_path

    return image_index


print("=" * 70)
print("IMAGE ↔ ANNOTATION MATCHING CHECK")
print("=" * 70)

print("\nBuilding image indexes...")

train_index = build_image_index(TRAIN_IMAGE_DIR)
val_index   = build_image_index(VAL_IMAGE_DIR)

print(f"Train images indexed: {len(train_index)}")
print(f"Val images indexed  : {len(val_index)}")


# ------------------------------------------------------------
# Check annotation IDs against image IDs
# ------------------------------------------------------------

def check_matching(annotation_file, image_index, split):

    total = 0
    matched = 0
    missing = []

    with open(annotation_file, "r") as f:

        for line in f:

            if not line.strip():
                continue

            annotation = json.loads(line)

            image_id = annotation["ID"]

            total += 1

            if image_id in image_index:
                matched += 1
            else:
                missing.append(image_id)

    print(f"\n{split.upper()} SET")
    print("-" * 50)
    print(f"Annotations : {total}")
    print(f"Matched     : {matched}")
    print(f"Missing     : {len(missing)}")

    if missing:
        print("\nFirst 10 missing image IDs:")

        for image_id in missing[:10]:
            print(" ", image_id)

    return total, matched, missing


train_total, train_matched, train_missing = check_matching(
    TRAIN_ANN,
    train_index,
    "train"
)

val_total, val_matched, val_missing = check_matching(
    VAL_ANN,
    val_index,
    "validation"
)


# ------------------------------------------------------------
# Final verification
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL MATCHING RESULT")
print("=" * 70)

if (
    train_total == train_matched
    and val_total == val_matched
):

    print("\n✅ PERFECT MATCH!")
    print("Every annotation has a corresponding image.")
    print("\nWe are ready for:")
    print("CrowdHuman → YOLO format conversion 🚀")

else:

    print("\n⚠️ MATCHING PROBLEM DETECTED")

    print(f"Train missing: {len(train_missing)}")
    print(f"Val missing  : {len(val_missing)}")

print("=" * 70)

In [ ]:
# =====================================================================
# CROWD DETECTION PROJECT — V1 ML PHASE
# CELL 2: FIRST REAL YOLO26m INFERENCE TEST
# =====================================================================
#
# PURPOSE:
#   Verify that our trained YOLO26m model can actually detect people
#   correctly on CrowdHuman validation images.
#
# THIS CELL DOES NOT TRAIN ANYTHING.
# It only performs inference/testing.
# =====================================================================

from pathlib import Path
import random
import time
import statistics
import os

from ultralytics import YOLO
from IPython.display import display
from PIL import Image

print("=" * 75)
print(" CROWD DETECTION PROJECT — V1 ML PHASE")
print(" CELL 2: FIRST REAL YOLO26m INFERENCE TEST")
print("=" * 75)

# ---------------------------------------------------------------------
# STEP 1 — Check model
# ---------------------------------------------------------------------

print("\n[1/8] Checking trained model...")
print("-" * 75)

MODEL_PATH = Path("/content/best.pt")

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"❌ Trained model not found at:\n{MODEL_PATH}\n"
        "Please upload best.pt again."
    )

print(f"✅ Model found:")
print(f"   {MODEL_PATH}")

print(f"   Size: {MODEL_PATH.stat().st_size / (1024**2):.2f} MB")

# ---------------------------------------------------------------------
# STEP 2 — Load model
# ---------------------------------------------------------------------

print("\n[2/8] Loading YOLO26m...")
print("-" * 75)

model = YOLO(str(MODEL_PATH))

print("✅ YOLO26m loaded")
print(f"   Task  : {model.task}")
print(f"   Class : {model.names}")

# ---------------------------------------------------------------------
# STEP 3 — Find CrowdHuman validation images
# ---------------------------------------------------------------------

print("\n[3/8] Looking for CrowdHuman validation images...")
print("-" * 75)

possible_val_dirs = [
    Path("/content/crowdhuman_yolo/images/val"),
    Path("/content/CrowdHuman/val_images/Images"),
    Path("/content/CrowdHuman/val/Images"),
]

VAL_DIR = None

for directory in possible_val_dirs:
    print(f"Checking: {directory}")

    if directory.exists():
        images_here = list(directory.glob("*.jpg"))

        if len(images_here) > 0:
            VAL_DIR = directory
            print(f"   ✅ Found {len(images_here):,} JPG images")
            break

if VAL_DIR is None:
    raise FileNotFoundError(
        "\n❌ Could not find CrowdHuman validation images.\n"
        "Expected one of:\n"
        + "\n".join(str(x) for x in possible_val_dirs)
    )

print(f"\n✅ Validation image directory:")
print(f"   {VAL_DIR}")

# ---------------------------------------------------------------------
# STEP 4 — Select test images
# ---------------------------------------------------------------------

print("\n[4/8] Selecting validation images...")
print("-" * 75)

all_images = sorted(VAL_DIR.glob("*.jpg"))

if len(all_images) == 0:
    raise RuntimeError("❌ No JPG images found.")

# Fixed seed makes our selection reproducible.
random.seed(42)

# We don't want to process hundreds of images yet.
# First, let's inspect a small sample manually.
NUM_TEST_IMAGES = 8

test_images = random.sample(
    all_images,
    min(NUM_TEST_IMAGES, len(all_images))
)

print(f"Total validation images available : {len(all_images):,}")
print(f"Images selected for this test     : {len(test_images)}")
print("\nSelected images:")

for i, image_path in enumerate(test_images, start=1):
    print(f"   {i}. {image_path.name}")

# ---------------------------------------------------------------------
# STEP 5 — Create output directory
# ---------------------------------------------------------------------

print("\n[5/8] Preparing inference output directory...")
print("-" * 75)

OUTPUT_DIR = Path("/content/inference_test")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"✅ Output directory:")
print(f"   {OUTPUT_DIR}")

# ---------------------------------------------------------------------
# STEP 6 — Run YOLO inference
# ---------------------------------------------------------------------

print("\n[6/8] Running YOLO26m inference...")
print("-" * 75)

print("Configuration:")
print("   Device       : GPU (Tesla T4)")
print("   Image size   : 640")
print("   Confidence   : 0.25")
print("   IoU          : 0.70")
print("   Max detections: 500")
print()

all_detection_counts = []
all_confidences = []
all_inference_times = []

results_data = []

for index, image_path in enumerate(test_images, start=1):

    print(f"\n--- Image {index}/{len(test_images)} ---")
    print(f"File: {image_path.name}")

    # ---------------------------------------------------------------
    # Open image to get original dimensions
    # ---------------------------------------------------------------

    with Image.open(image_path) as img:
        width, height = img.size

    print(f"Original size: {width} × {height}")

    # ---------------------------------------------------------------
    # Measure inference time
    # ---------------------------------------------------------------

    start_time = time.perf_counter()

    results = model.predict(
        source=str(image_path),
        imgsz=640,
        conf=0.25,
        iou=0.70,
        max_det=500,
        device=0,
        verbose=False,
        save=False
    )

    elapsed = time.perf_counter() - start_time

    result = results[0]

    # ---------------------------------------------------------------
    # Extract detections
    # ---------------------------------------------------------------

    boxes = result.boxes

    if boxes is not None and len(boxes) > 0:

        # Confidence values
        confidences = boxes.conf.detach().cpu().numpy().tolist()

        # Class IDs
        class_ids = boxes.cls.detach().cpu().numpy().tolist()

        # Count only class 0 = person
        person_confidences = [
            conf
            for conf, cls_id in zip(confidences, class_ids)
            if int(cls_id) == 0
        ]

    else:
        person_confidences = []

    person_count = len(person_confidences)

    # ---------------------------------------------------------------
    # Store statistics
    # ---------------------------------------------------------------

    all_detection_counts.append(person_count)
    all_confidences.extend(person_confidences)
    all_inference_times.append(elapsed)

    results_data.append({
        "image": image_path.name,
        "width": width,
        "height": height,
        "person_count": person_count,
        "avg_confidence": (
            statistics.mean(person_confidences)
            if person_confidences else 0.0
        ),
        "min_confidence": (
            min(person_confidences)
            if person_confidences else 0.0
        ),
        "max_confidence": (
            max(person_confidences)
            if person_confidences else 0.0
        ),
        "inference_time_sec": elapsed
    })

    # ---------------------------------------------------------------
    # Print image-level results
    # ---------------------------------------------------------------

    print(f"People detected : {person_count}")

    if person_confidences:
        print(
            f"Confidence      : "
            f"min={min(person_confidences):.3f}, "
            f"avg={statistics.mean(person_confidences):.3f}, "
            f"max={max(person_confidences):.3f}"
        )
    else:
        print("Confidence      : No detections")

    print(f"Inference time  : {elapsed:.4f} sec")
    print(f"Approx FPS      : {1 / elapsed:.2f}")

    # ---------------------------------------------------------------
    # Save annotated image
    # ---------------------------------------------------------------

    annotated = result.plot()

    output_path = OUTPUT_DIR / image_path.name

    Image.fromarray(annotated[:, :, ::-1]).save(output_path)

    print(f"Saved result    : {output_path}")

# ---------------------------------------------------------------------
# STEP 7 — Overall statistics
# ---------------------------------------------------------------------

print("\n\n[7/8] Calculating overall inference statistics...")
print("-" * 75)

print(f"Images processed       : {len(results_data)}")

print(
    f"Total people detected : "
    f"{sum(all_detection_counts):,}"
)

print(
    f"Average people/image  : "
    f"{statistics.mean(all_detection_counts):.2f}"
)

print(
    f"Minimum people/image  : "
    f"{min(all_detection_counts)}"
)

print(
    f"Maximum people/image  : "
    f"{max(all_detection_counts)}"
)

if all_confidences:
    print(
        f"\nDetection confidence:"
    )
    print(
        f"   Minimum : {min(all_confidences):.3f}"
    )
    print(
        f"   Average : {statistics.mean(all_confidences):.3f}"
    )
    print(
        f"   Maximum : {max(all_confidences):.3f}"
    )

print("\nInference speed:")
print(
    f"   Average time : "
    f"{statistics.mean(all_inference_times):.4f} sec/image"
)

print(
    f"   Approx FPS   : "
    f"{1 / statistics.mean(all_inference_times):.2f}"
)

# ---------------------------------------------------------------------
# STEP 8 — Display annotated images
# ---------------------------------------------------------------------

print("\n[8/8] Displaying annotated results...")
print("-" * 75)

print("The following images show what YOLO26m detected.")
print("Green/colored bounding boxes = detected people.")
print()

for image_path in test_images:

    output_path = OUTPUT_DIR / image_path.name

    if output_path.exists():

        print(f"\n📷 {image_path.name}")

        img = Image.open(output_path)

        # Display resized image if it is extremely large.
        display_img = img.copy()
        display_img.thumbnail((1200, 900))

        display(display_img)

# ---------------------------------------------------------------------
# FINAL SUMMARY
# ---------------------------------------------------------------------

print("\n" + "=" * 75)
print(" CELL 2 COMPLETE — FIRST INFERENCE TEST FINISHED")
print("=" * 75)

print("\n✅ What we have verified:")
print("   1. YOLO26m successfully performed inference.")
print("   2. Person detections were extracted.")
print("   3. Detection confidence was measured.")
print("   4. Inference speed was measured.")
print("   5. Annotated images were saved.")
print()
print(f"📁 Annotated images:")
print(f"   {OUTPUT_DIR}")
print()
print("📌 IMPORTANT:")
print("   This is only a small inference test.")
print("   We are NOT creating the ML CSV yet.")
print("   We first need to inspect these detections.")
print("=" * 75)

In [ ]:
# ============================================================
# CELL 5 — CROWDHUMAN → YOLO FORMAT CONVERSION
# ============================================================
#
# CrowdHuman provides bounding boxes in this format:
#
#     [x, y, width, height]
#
# YOLO requires:
#
#     class_id center_x center_y width height
#
# All coordinates except class_id must be normalized to 0-1.
#
# We use:
#
#     class_id = 0
#
# because our only class is:
#
#     0 = person
#
# We use CrowdHuman's "fbox" because it represents the
# full-body bounding box of a person.
# ============================================================

import json
import cv2
import shutil
from pathlib import Path
from tqdm.auto import tqdm


# ------------------------------------------------------------
# 1. PATH CONFIGURATION
# ------------------------------------------------------------

DATASET = Path("/content/CrowdHuman")

OUTPUT = Path("/content/crowdhuman_yolo")

TRAIN_IMAGES = DATASET / "train_images"
VAL_IMAGES   = DATASET / "val_images"

TRAIN_ANN = DATASET / "annotation_train.odgt"
VAL_ANN   = DATASET / "annotation_val.odgt"


print("=" * 70)
print("CROWDHUMAN → YOLO DATASET CONVERSION")
print("=" * 70)

print("\nSource dataset:")
print(f"  {DATASET}")

print("\nYOLO dataset:")
print(f"  {OUTPUT}")


# ------------------------------------------------------------
# 2. CREATE YOLO DIRECTORY STRUCTURE
# ------------------------------------------------------------

print("\n[1] Creating YOLO directory structure...")

for split in ["train", "val"]:

    (OUTPUT / "images" / split).mkdir(
        parents=True,
        exist_ok=True
    )

    (OUTPUT / "labels" / split).mkdir(
        parents=True,
        exist_ok=True
    )

print("✅ Directory structure created")


# ------------------------------------------------------------
# 3. BUILD IMAGE INDEX
# ------------------------------------------------------------
#
# Instead of searching the filesystem for every annotation,
# we create a dictionary:
#
#     image_id → image_path
#
# Example:
#
#     "284193,faa9000f2678b5e"
#          ↓
#     ".../Images/284193,faa9000f2678b5e.jpg"
#
# ------------------------------------------------------------

def build_image_index(directory):

    print(f"\nIndexing images in:")
    print(f"  {directory}")

    image_index = {}

    for image_path in directory.rglob("*"):

        if image_path.suffix.lower() in {
            ".jpg",
            ".jpeg",
            ".png"
        }:

            image_id = image_path.stem

            image_index[image_id] = image_path

    print(f"✅ Found {len(image_index)} images")

    return image_index


train_index = build_image_index(TRAIN_IMAGES)
val_index   = build_image_index(VAL_IMAGES)


# ------------------------------------------------------------
# 4. CONVERT ANNOTATIONS
# ------------------------------------------------------------

def convert_split(annotation_file, image_index, split):

    print("\n" + "=" * 70)
    print(f"PROCESSING {split.upper()} SET")
    print("=" * 70)

    processed_images = 0
    missing_images = 0
    invalid_boxes = 0
    person_boxes = 0

    # --------------------------------------------------------
    # Read annotation file
    # --------------------------------------------------------

    with open(annotation_file, "r") as f:

        lines = f.readlines()

    print(f"\nAnnotations found: {len(lines)}")

    # --------------------------------------------------------
    # Process every image
    # --------------------------------------------------------

    for line in tqdm(
        lines,
        desc=f"Converting {split}"
    ):

        if not line.strip():
            continue

        # ----------------------------------------------------
        # Parse JSON annotation
        # ----------------------------------------------------

        annotation = json.loads(line)

        image_id = annotation["ID"]

        # ----------------------------------------------------
        # Find corresponding image
        # ----------------------------------------------------

        image_path = image_index.get(image_id)

        if image_path is None:

            missing_images += 1

            continue

        # ----------------------------------------------------
        # Read image
        # ----------------------------------------------------

        image = cv2.imread(str(image_path))

        if image is None:

            print(
                f"\n⚠️ Could not read image: "
                f"{image_path}"
            )

            continue

        image_height, image_width = image.shape[:2]

        # ----------------------------------------------------
        # Output paths
        # ----------------------------------------------------

        output_image = (
            OUTPUT
            / "images"
            / split
            / f"{image_id}.jpg"
        )

        output_label = (
            OUTPUT
            / "labels"
            / split
            / f"{image_id}.txt"
        )

        # ----------------------------------------------------
        # Copy image into YOLO dataset
        # ----------------------------------------------------

        #
        # We use shutil.copy2 instead of rewriting the image.
        # This preserves the original image quality.
        #

        shutil.copy2(
            image_path,
            output_image
        )

        # ----------------------------------------------------
        # Convert every person bounding box
        # ----------------------------------------------------

        yolo_labels = []

        for gtbox in annotation.get("gtboxes", []):

            # ------------------------------------------------
            # Keep only actual persons
            # ------------------------------------------------

            if gtbox.get("tag") != "person":
                continue

            # ------------------------------------------------
            # Get full-body bounding box
            # ------------------------------------------------

            fbox = gtbox.get("fbox")

            if fbox is None or len(fbox) != 4:

                invalid_boxes += 1

                continue

            x, y, box_width, box_height = fbox

            # ------------------------------------------------
            # Validate bounding box
            # ------------------------------------------------

            if box_width <= 0 or box_height <= 0:

                invalid_boxes += 1

                continue

            # ------------------------------------------------
            # CrowdHuman XYWH
            #
            #     x = left
            #     y = top
            #     w = width
            #     h = height
            #
            # YOLO needs center coordinates.
            # ------------------------------------------------

            center_x = x + (box_width / 2)
            center_y = y + (box_height / 2)

            # ------------------------------------------------
            # Normalize to 0-1
            # ------------------------------------------------

            center_x = center_x / image_width
            center_y = center_y / image_height

            box_width = box_width / image_width
            box_height = box_height / image_height

            # ------------------------------------------------
            # Sanity check
            # ------------------------------------------------

            if not (
                0 <= center_x <= 1
                and 0 <= center_y <= 1
                and 0 < box_width <= 1
                and 0 < box_height <= 1
            ):

                invalid_boxes += 1

                continue

            # ------------------------------------------------
            # YOLO label
            #
            # 0 = person
            #
            # Format:
            #
            # 0 center_x center_y width height
            # ------------------------------------------------

            yolo_labels.append(
                f"0 "
                f"{center_x:.6f} "
                f"{center_y:.6f} "
                f"{box_width:.6f} "
                f"{box_height:.6f}"
            )

            person_boxes += 1

        # ----------------------------------------------------
        # Save YOLO label file
        # ----------------------------------------------------

        with open(output_label, "w") as f:

            f.write(
                "\n".join(yolo_labels)
            )

        processed_images += 1

    # --------------------------------------------------------
    # Print split summary
    # --------------------------------------------------------

    print("\n" + "-" * 70)

    print(f"{split.upper()} SUMMARY")

    print("-" * 70)

    print(
        f"Images processed : {processed_images}"
    )

    print(
        f"Missing images   : {missing_images}"
    )

    print(
        f"Person boxes     : {person_boxes}"
    )

    print(
        f"Invalid boxes    : {invalid_boxes}"
    )


# ------------------------------------------------------------
# 5. CONVERT TRAINING SET
# ------------------------------------------------------------

convert_split(
    TRAIN_ANN,
    train_index,
    "train"
)


# ------------------------------------------------------------
# 6. CONVERT VALIDATION SET
# ------------------------------------------------------------

convert_split(
    VAL_ANN,
    val_index,
    "val"
)


# ------------------------------------------------------------
# 7. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CONVERSION COMPLETE ✅")
print("=" * 70)

print("\nYOLO dataset created at:")

print(f"  {OUTPUT}")

print("\nStructure:")

print("""
crowdhuman_yolo/
├── images/
│   ├── train/
│   └── val/
└── labels/
    ├── train/
    └── val/
""")

print("=" * 70)

In [ ]:
# CELL 6 — VISUAL VERIFICATION OF YOLO LABELS
# ============================================================
#
# This is our final sanity check before training.
#
# We will:
#   1. Pick a random training image
#   2. Load its YOLO label file
#   3. Convert YOLO coordinates back to pixels
#   4. Draw the person bounding boxes
#   5. Display the image
#
# We expect to see GREEN/colored rectangles around people.
# ============================================================

import random
import cv2
import matplotlib.pyplot as plt
from pathlib import Path


# ------------------------------------------------------------
# Dataset paths
# ------------------------------------------------------------

YOLO_DATASET = Path("/content/crowdhuman_yolo")

IMAGE_DIR = YOLO_DATASET / "images" / "train"
LABEL_DIR = YOLO_DATASET / "labels" / "train"


# ------------------------------------------------------------
# Find images
# ------------------------------------------------------------

image_files = list(IMAGE_DIR.glob("*.jpg"))

print("=" * 70)
print("YOLO LABEL VISUAL VERIFICATION")
print("=" * 70)

print(f"\nTraining images available: {len(image_files)}")


# ------------------------------------------------------------
# Select random image
# ------------------------------------------------------------

random_image = random.choice(image_files)

image_id = random_image.stem

label_file = LABEL_DIR / f"{image_id}.txt"

print("\nSelected image:")
print(f"  {random_image.name}")

print("\nCorresponding label:")
print(f"  {label_file.name}")


# ------------------------------------------------------------
# Read image
# ------------------------------------------------------------

image = cv2.imread(str(random_image))

if image is None:
    raise RuntimeError("❌ Could not read image!")

image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

image_height, image_width = image.shape[:2]

print("\nImage dimensions:")
print(f"  Width  : {image_width}")
print(f"  Height : {image_height}")


# ------------------------------------------------------------
# Read YOLO labels
# ------------------------------------------------------------

if not label_file.exists():
    raise RuntimeError("❌ Label file does not exist!")

with open(label_file, "r") as f:
    lines = [line.strip() for line in f if line.strip()]

print("\nNumber of YOLO annotations:")
print(f"  {len(lines)}")


# ------------------------------------------------------------
# Draw bounding boxes
# ------------------------------------------------------------

boxes_drawn = 0

for line in lines:

    values = line.split()

    if len(values) != 5:
        continue

    class_id, center_x, center_y, width, height = map(
        float,
        values
    )

    # --------------------------------------------------------
    # YOLO normalized coordinates → pixel coordinates
    # --------------------------------------------------------

    center_x *= image_width
    center_y *= image_height

    width *= image_width
    height *= image_height

    x1 = int(center_x - width / 2)
    y1 = int(center_y - height / 2)

    x2 = int(center_x + width / 2)
    y2 = int(center_y + height / 2)

    # Keep coordinates inside image
    x1 = max(0, min(image_width - 1, x1))
    y1 = max(0, min(image_height - 1, y1))
    x2 = max(0, min(image_width - 1, x2))
    y2 = max(0, min(image_height - 1, y2))

    # --------------------------------------------------------
    # Draw bounding box
    # --------------------------------------------------------

    cv2.rectangle(
        image,
        (x1, y1),
        (x2, y2),
        (0, 255, 0),
        2
    )

    # Label
    cv2.putText(
        image,
        "person",
        (x1, max(15, y1 - 5)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (0, 255, 0),
        1,
        cv2.LINE_AA
    )

    boxes_drawn += 1


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print(f"\nBounding boxes drawn: {boxes_drawn}")

print("\nDisplaying image...")

plt.figure(figsize=(14, 9))
plt.imshow(image)
plt.axis("off")
plt.title(
    f"CrowdHuman — {image_id}\n"
    f"Person boxes: {boxes_drawn}"
)
plt.show()

In [ ]:
# ============================================================
# CELL 7 — CREATE YOLO DATASET CONFIGURATION
# ============================================================
#
# This file tells Ultralytics YOLO how our CrowdHuman dataset
# is organized.
#
# Dataset:
#
# /content/crowdhuman_yolo/
# ├── images/
# │   ├── train/
# │   └── val/
# │
# └── labels/
#     ├── train/
#     └── val/
#
# We have ONLY ONE detection class:
#
#     0 = person
#
# ============================================================

from pathlib import Path

# ------------------------------------------------------------
# Dataset location
# ------------------------------------------------------------

YOLO_DATASET = Path("/content/crowdhuman_yolo")

DATA_YAML = YOLO_DATASET / "data.yaml"


print("=" * 70)
print("CREATING YOLO DATASET CONFIGURATION")
print("=" * 70)


# ------------------------------------------------------------
# Check required directories
# ------------------------------------------------------------

required_directories = [
    YOLO_DATASET / "images" / "train",
    YOLO_DATASET / "images" / "val",
    YOLO_DATASET / "labels" / "train",
    YOLO_DATASET / "labels" / "val",
]

print("\n[1] Checking dataset directories...\n")

for directory in required_directories:

    if directory.exists():

        print(f"✅ {directory}")

    else:

        print(f"❌ MISSING: {directory}")

        raise FileNotFoundError(
            f"Required directory does not exist: {directory}"
        )


# ------------------------------------------------------------
# Count images and labels
# ------------------------------------------------------------

print("\n[2] Counting images and labels...\n")


def count_files(directory, extensions):

    return sum(
        1
        for file in directory.iterdir()
        if file.is_file()
        and file.suffix.lower() in extensions
    )


train_images = count_files(
    YOLO_DATASET / "images" / "train",
    {".jpg", ".jpeg", ".png"}
)

val_images = count_files(
    YOLO_DATASET / "images" / "val",
    {".jpg", ".jpeg", ".png"}
)

train_labels = count_files(
    YOLO_DATASET / "labels" / "train",
    {".txt"}
)

val_labels = count_files(
    YOLO_DATASET / "labels" / "val",
    {".txt"}
)


print(f"Training images : {train_images}")
print(f"Training labels : {train_labels}")

print(f"\nValidation images : {val_images}")
print(f"Validation labels : {val_labels}")


# ------------------------------------------------------------
# Create data.yaml
# ------------------------------------------------------------

print("\n[3] Creating data.yaml...\n")


yaml_content = """path: /content/crowdhuman_yolo

train: images/train
val: images/val

names:
  0: person
"""


with open(DATA_YAML, "w") as file:
    file.write(yaml_content)


print("✅ data.yaml created")


# ------------------------------------------------------------
# Display YAML
# ------------------------------------------------------------

print("\n[4] Configuration:\n")
print("-" * 50)

print(DATA_YAML.read_text())

print("-" * 50)


# ------------------------------------------------------------
# Final checks
# ------------------------------------------------------------

print("\n[5] Final dataset check...\n")

checks_passed = True


if train_images != 15000:
    print(
        f"⚠️ Expected 15000 training images, "
        f"found {train_images}"
    )
    checks_passed = False
else:
    print("✅ Training image count = 15000")


if val_images != 4370:
    print(
        f"⚠️ Expected 4370 validation images, "
        f"found {val_images}"
    )
    checks_passed = False
else:
    print("✅ Validation image count = 4370")


if train_labels != 15000:
    print(
        f"⚠️ Expected 15000 training labels, "
        f"found {train_labels}"
    )
    checks_passed = False
else:
    print("✅ Training label count = 15000")


if val_labels != 4370:
    print(
        f"⚠️ Expected 4370 validation labels, "
        f"found {val_labels}"
    )
    checks_passed = False
else:
    print("✅ Validation label count = 4370")


print("\n" + "=" * 70)

if checks_passed:

    print("🎉 DATASET CONFIGURATION PASSED!")

    print("\nReady for YOLO validation/training.")

else:

    print("⚠️ SOME CHECKS FAILED.")

print("=" * 70)

In [ ]:
# ============================================================
# CELL 7 — CREATE YOLO DATASET CONFIGURATION
# ============================================================
#
# This file tells Ultralytics YOLO how our CrowdHuman dataset
# is organized.
#
# Dataset:
#
# /content/crowdhuman_yolo/
# ├── images/
# │   ├── train/
# │   └── val/
# │
# └── labels/
#     ├── train/
#     └── val/
#
# We have ONLY ONE detection class:
#
#     0 = person
#
# ============================================================

from pathlib import Path

# ------------------------------------------------------------
# Dataset location
# ------------------------------------------------------------

YOLO_DATASET = Path("/content/crowdhuman_yolo")

DATA_YAML = YOLO_DATASET / "data.yaml"


print("=" * 70)
print("CREATING YOLO DATASET CONFIGURATION")
print("=" * 70)


# ------------------------------------------------------------
# Check required directories
# ------------------------------------------------------------

required_directories = [
    YOLO_DATASET / "images" / "train",
    YOLO_DATASET / "images" / "val",
    YOLO_DATASET / "labels" / "train",
    YOLO_DATASET / "labels" / "val",
]

print("\n[1] Checking dataset directories...\n")

for directory in required_directories:

    if directory.exists():

        print(f"✅ {directory}")

    else:

        print(f"❌ MISSING: {directory}")

        raise FileNotFoundError(
            f"Required directory does not exist: {directory}"
        )


# ------------------------------------------------------------
# Count images and labels
# ------------------------------------------------------------

print("\n[2] Counting images and labels...\n")


def count_files(directory, extensions):

    return sum(
        1
        for file in directory.iterdir()
        if file.is_file()
        and file.suffix.lower() in extensions
    )


train_images = count_files(
    YOLO_DATASET / "images" / "train",
    {".jpg", ".jpeg", ".png"}
)

val_images = count_files(
    YOLO_DATASET / "images" / "val",
    {".jpg", ".jpeg", ".png"}
)

train_labels = count_files(
    YOLO_DATASET / "labels" / "train",
    {".txt"}
)

val_labels = count_files(
    YOLO_DATASET / "labels" / "val",
    {".txt"}
)


print(f"Training images : {train_images}")
print(f"Training labels : {train_labels}")

print(f"\nValidation images : {val_images}")
print(f"Validation labels : {val_labels}")


# ------------------------------------------------------------
# Create data.yaml
# ------------------------------------------------------------

print("\n[3] Creating data.yaml...\n")


yaml_content = """path: /content/crowdhuman_yolo

train: images/train
val: images/val

names:
  0: person
"""


with open(DATA_YAML, "w") as file:
    file.write(yaml_content)


print("✅ data.yaml created")


# ------------------------------------------------------------
# Display YAML
# ------------------------------------------------------------

print("\n[4] Configuration:\n")
print("-" * 50)

print(DATA_YAML.read_text())

print("-" * 50)


# ------------------------------------------------------------
# Final checks
# ------------------------------------------------------------

print("\n[5] Final dataset check...\n")

checks_passed = True


if train_images != 15000:
    print(
        f"⚠️ Expected 15000 training images, "
        f"found {train_images}"
    )
    checks_passed = False
else:
    print("✅ Training image count = 15000")


if val_images != 4370:
    print(
        f"⚠️ Expected 4370 validation images, "
        f"found {val_images}"
    )
    checks_passed = False
else:
    print("✅ Validation image count = 4370")


if train_labels != 15000:
    print(
        f"⚠️ Expected 15000 training labels, "
        f"found {train_labels}"
    )
    checks_passed = False
else:
    print("✅ Training label count = 15000")


if val_labels != 4370:
    print(
        f"⚠️ Expected 4370 validation labels, "
        f"found {val_labels}"
    )
    checks_passed = False
else:
    print("✅ Validation label count = 4370")


print("\n" + "=" * 70)

if checks_passed:

    print("🎉 DATASET CONFIGURATION PASSED!")

    print("\nReady for YOLO validation/training.")

else:

    print("⚠️ SOME CHECKS FAILED.")

print("=" * 70)

In [ ]:
# =====================================================================
# CROWD DETECTION PROJECT — V1 ML PHASE
# CELL 3: FULL YOLO26m VALIDATION
# =====================================================================
#
# PURPOSE:
#   Evaluate the final trained YOLO26m model on the complete
#   CrowdHuman validation dataset.
#
# IMPORTANT:
#   - No training happens here.
#   - We are only evaluating best.pt.
#   - This gives us the official detector metrics we will record
#     before moving into the Random Forest feature-extraction stage.
# =====================================================================

from pathlib import Path
import time
import torch
from ultralytics import YOLO

print("=" * 80)
print(" CROWD DETECTION PROJECT — V1 ML PHASE")
print(" CELL 3: FULL YOLO26m VALIDATION")
print("=" * 80)

# ---------------------------------------------------------------------
# STEP 1 — Environment
# ---------------------------------------------------------------------

print("\n[1/7] Checking runtime environment...")
print("-" * 80)

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"CUDA version    : {torch.version.cuda}")
    device = 0
else:
    print("⚠️ GPU is not available.")
    print("Validation will run on CPU.")
    device = "cpu"

# ---------------------------------------------------------------------
# STEP 2 — Model
# ---------------------------------------------------------------------

print("\n[2/7] Loading trained YOLO26m model...")
print("-" * 80)

MODEL_PATH = Path("/content/best.pt")

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"❌ best.pt not found at:\n{MODEL_PATH}"
    )

print(f"Model path : {MODEL_PATH}")
print(f"Model size : {MODEL_PATH.stat().st_size / (1024**2):.2f} MB")

model = YOLO(str(MODEL_PATH))

print("\n✅ Model loaded successfully")
print(f"Task       : {model.task}")
print(f"Classes    : {model.names}")

# ---------------------------------------------------------------------
# STEP 3 — Dataset configuration
# ---------------------------------------------------------------------

print("\n[3/7] Checking CrowdHuman YOLO dataset...")
print("-" * 80)

DATA_YAML = Path("/content/crowdhuman_yolo/data.yaml")

if not DATA_YAML.exists():
    raise FileNotFoundError(
        f"❌ data.yaml not found at:\n{DATA_YAML}\n"
        "Please recreate the YOLO dataset first."
    )

print(f"Dataset YAML : {DATA_YAML}")

print("\nDataset YAML contents:")
print("-" * 40)

with open(DATA_YAML, "r") as f:
    yaml_text = f.read()

print(yaml_text)

# ---------------------------------------------------------------------
# STEP 4 — Count validation data
# ---------------------------------------------------------------------

print("\n[4/7] Checking validation images and labels...")
print("-" * 80)

VAL_IMAGES = Path("/content/crowdhuman_yolo/images/val")
VAL_LABELS = Path("/content/crowdhuman_yolo/labels/val")

if not VAL_IMAGES.exists():
    raise FileNotFoundError(
        f"❌ Validation image directory not found:\n{VAL_IMAGES}"
    )

if not VAL_LABELS.exists():
    raise FileNotFoundError(
        f"❌ Validation label directory not found:\n{VAL_LABELS}"
    )

val_images = sorted(VAL_IMAGES.glob("*.jpg"))
val_labels = sorted(VAL_LABELS.glob("*.txt"))

print(f"Validation images : {len(val_images):,}")
print(f"Validation labels : {len(val_labels):,}")

# Check image/label filename matching.

image_stems = {p.stem for p in val_images}
label_stems = {p.stem for p in val_labels}

missing_labels = image_stems - label_stems
extra_labels = label_stems - image_stems

print(f"Images without labels : {len(missing_labels):,}")
print(f"Labels without images : {len(extra_labels):,}")

if len(missing_labels) == 0 and len(extra_labels) == 0:
    print("✅ Image/label filenames match perfectly.")
else:
    print("⚠️ There is an image/label mismatch.")

    if missing_labels:
        print("\nFirst few missing labels:")
        for name in list(sorted(missing_labels))[:10]:
            print(f"   {name}")

    if extra_labels:
        print("\nFirst few extra labels:")
        for name in list(sorted(extra_labels))[:10]:
            print(f"   {name}")

# ---------------------------------------------------------------------
# STEP 5 — Count ground-truth person boxes
# ---------------------------------------------------------------------

print("\n[5/7] Inspecting ground-truth labels...")
print("-" * 80)

total_gt_boxes = 0
empty_label_files = 0
invalid_label_lines = 0

for label_file in val_labels:

    with open(label_file, "r") as f:
        lines = [line.strip() for line in f if line.strip()]

    if len(lines) == 0:
        empty_label_files += 1
        continue

    for line in lines:

        parts = line.split()

        if len(parts) != 5:
            invalid_label_lines += 1
            continue

        try:
            class_id = int(parts[0])
            x = float(parts[1])
            y = float(parts[2])
            w = float(parts[3])
            h = float(parts[4])

            # Basic YOLO label sanity check.
            if class_id != 0:
                invalid_label_lines += 1
                continue

            if not (
                0 <= x <= 1 and
                0 <= y <= 1 and
                0 < w <= 1 and
                0 < h <= 1
            ):
                invalid_label_lines += 1
                continue

            total_gt_boxes += 1

        except ValueError:
            invalid_label_lines += 1

print(f"Ground-truth person boxes : {total_gt_boxes:,}")
print(f"Empty label files          : {empty_label_files:,}")
print(f"Invalid label lines        : {invalid_label_lines:,}")

if invalid_label_lines == 0:
    print("✅ Ground-truth labels passed sanity checks.")
else:
    print("⚠️ Some invalid label lines were found.")

# ---------------------------------------------------------------------
# STEP 6 — Run complete validation
# ---------------------------------------------------------------------

print("\n[6/7] Running FULL YOLO26m validation...")
print("-" * 80)

print("This will evaluate all validation images.")
print("No model weights will be changed.")
print()
print("Configuration:")
print("   Image size       : 640")
print("   Confidence       : Default validation threshold")
print("   IoU threshold    : Default validation setting")
print("   Max detections   : 500")
print(f"   Device           : {device}")
print(f"   Validation imgs  : {len(val_images):,}")
print()

start_time = time.perf_counter()

validation_results = model.val(
    data=str(DATA_YAML),
    imgsz=640,
    batch=8,
    device=device,
    max_det=500,
    plots=True,
    verbose=True
)

elapsed = time.perf_counter() - start_time

print("\nValidation execution time:")
print(f"   Total time : {elapsed / 60:.2f} minutes")

# ---------------------------------------------------------------------
# STEP 7 — Extract and display final metrics
# ---------------------------------------------------------------------

print("\n[7/7] Extracting final YOLO metrics...")
print("-" * 80)

metrics = validation_results.box

precision = float(metrics.mp)
recall = float(metrics.mr)
map50 = float(metrics.map50)
map5095 = float(metrics.map)

print("\n" + "=" * 80)
print(" FINAL YOLO26m VALIDATION RESULTS")
print("=" * 80)

print(f"\nPrecision       : {precision:.4f}  ({precision * 100:.2f}%)")
print(f"Recall          : {recall:.4f}  ({recall * 100:.2f}%)")
print(f"mAP@50          : {map50:.4f}  ({map50 * 100:.2f}%)")
print(f"mAP@50-95       : {map5095:.4f}  ({map5095 * 100:.2f}%)")

print("\n" + "-" * 80)
print("WHAT THESE NUMBERS MEAN")
print("-" * 80)

print("""
Precision:
    Of the people YOLO predicted, how many were correct?

Recall:
    Of the people actually present, how many did YOLO find?

mAP@50:
    Detection performance when a predicted box is considered correct
    at IoU = 0.50.

mAP@50-95:
    A stricter localization metric averaged across IoU thresholds
    from 0.50 to 0.95.
""")

print("=" * 80)
print(" CELL 3 COMPLETE")
print("=" * 80)

print("\n✅ The trained YOLO26m detector has now been evaluated.")
print("✅ These metrics will be our V1 detector baseline.")
print("➡️  After reviewing this output, we will move to feature extraction.")
print("=" * 80)

In [ ]:
# =====================================================================
# CROWD DETECTION PROJECT — V1 ML PHASE
# CELL 4: PILOT FEATURE EXTRACTION
# =====================================================================
#
# PURPOSE:
#   Convert YOLO26m person detections into numerical scene-level
#   features that can later be used by Random Forest.
#
# IMPORTANT:
#   - NO Random Forest yet.
#   - NO CROWD / NOT-CROWD labels yet.
#   - NO training happens here.
#   - We are testing the feature extraction pipeline on 100 images.
#
# OUTPUT:
#   /content/features_pilot_100.csv
#
# Each row = one image
# Each column = one numerical feature
# =====================================================================

from pathlib import Path
import time
import math
import statistics
import numpy as np
import pandas as pd
import torch

from ultralytics import YOLO

print("=" * 80)
print(" CROWD DETECTION PROJECT — V1 ML PHASE")
print(" CELL 4: PILOT FEATURE EXTRACTION")
print("=" * 80)

# ---------------------------------------------------------------------
# STEP 1 — Environment
# ---------------------------------------------------------------------

print("\n[1/9] Checking environment...")
print("-" * 80)

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    DEVICE = 0
else:
    print("⚠️ GPU unavailable — using CPU.")
    DEVICE = "cpu"

# ---------------------------------------------------------------------
# STEP 2 — Load trained model
# ---------------------------------------------------------------------

print("\n[2/9] Loading trained YOLO26m model...")
print("-" * 80)

MODEL_PATH = Path("/content/best.pt")

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"❌ Trained model not found:\n{MODEL_PATH}"
    )

model = YOLO(str(MODEL_PATH))

print("✅ Model loaded")
print(f"Model : {MODEL_PATH}")
print(f"Task  : {model.task}")
print(f"Class : {model.names}")

# ---------------------------------------------------------------------
# STEP 3 — Find validation images
# ---------------------------------------------------------------------

print("\n[3/9] Locating CrowdHuman validation images...")
print("-" * 80)

VAL_DIR = Path("/content/CrowdHuman/val_images/Images")

if not VAL_DIR.exists():
    # Fallback to YOLO validation directory.
    VAL_DIR = Path("/content/crowdhuman_yolo/images/val")

if not VAL_DIR.exists():
    raise FileNotFoundError(
        "❌ Could not find CrowdHuman validation images."
    )

all_images = sorted(VAL_DIR.glob("*.jpg"))

if len(all_images) == 0:
    raise RuntimeError(
        f"❌ No JPG images found in:\n{VAL_DIR}"
    )

print(f"Validation directory : {VAL_DIR}")
print(f"Total images found   : {len(all_images):,}")

# ---------------------------------------------------------------------
# STEP 4 — Select pilot dataset
# ---------------------------------------------------------------------

print("\n[4/9] Selecting pilot images...")
print("-" * 80)

# Fixed seed means we can reproduce the same pilot dataset later.
np.random.seed(42)

PILOT_SIZE = 100

if len(all_images) < PILOT_SIZE:
    test_images = all_images
else:
    selected_indices = np.random.choice(
        len(all_images),
        size=PILOT_SIZE,
        replace=False
    )

    test_images = [
        all_images[int(i)]
        for i in selected_indices
    ]

print(f"Pilot size          : {len(test_images)} images")
print("Random seed         : 42")
print("Replacement         : False")

print("\nFirst 10 selected images:")

for i, image_path in enumerate(test_images[:10], start=1):
    print(f"   {i:02d}. {image_path.name}")

if len(test_images) > 10:
    print(f"   ... and {len(test_images) - 10} more")

# ---------------------------------------------------------------------
# STEP 5 — Define feature extraction function
# ---------------------------------------------------------------------

print("\n[5/9] Preparing feature extraction...")
print("-" * 80)

print("""
The following scene-level features will be extracted:

BASIC DETECTION FEATURES
    person_count
    avg_confidence
    min_confidence
    max_confidence

SIZE / OCCUPANCY FEATURES
    total_bbox_area_ratio
    avg_bbox_area_ratio
    max_bbox_area_ratio
    avg_bbox_width_ratio
    avg_bbox_height_ratio

DENSITY FEATURES
    people_per_megapixel

SPATIAL FEATURES
    average_center_distance
    average_nearest_neighbor_distance

3 × 3 SPATIAL GRID
    People are divided according to their bounding-box centers
    into 9 regions of the image.

    +-------+-------+-------+
    |  G00  |  G01  |  G02  |
    +-------+-------+-------+
    |  G10  |  G11  |  G12  |
    +-------+-------+-------+
    |  G20  |  G21  |  G22  |
    +-------+-------+-------+

This allows Random Forest to learn not only "how many people",
but also "where the people are located".
""")

# ---------------------------------------------------------------------
# Helper function
# ---------------------------------------------------------------------

def extract_scene_features(result, image_width, image_height):
    """
    Convert YOLO detections for ONE image into scene-level
    numerical features.
    """

    # ---------------------------------------------------------------
    # Get person detections
    # ---------------------------------------------------------------

    boxes = result.boxes

    if boxes is None or len(boxes) == 0:

        return {
            "person_count": 0,

            "avg_confidence": 0.0,
            "min_confidence": 0.0,
            "max_confidence": 0.0,

            "total_bbox_area_ratio": 0.0,
            "avg_bbox_area_ratio": 0.0,
            "max_bbox_area_ratio": 0.0,

            "avg_bbox_width_ratio": 0.0,
            "avg_bbox_height_ratio": 0.0,

            "people_per_megapixel": 0.0,

            "average_center_distance": 0.0,
            "average_nearest_neighbor_distance": 0.0,

            "grid_00": 0,
            "grid_01": 0,
            "grid_02": 0,
            "grid_10": 0,
            "grid_11": 0,
            "grid_12": 0,
            "grid_20": 0,
            "grid_21": 0,
            "grid_22": 0,
        }

    # ---------------------------------------------------------------
    # Move YOLO tensors to CPU
    # ---------------------------------------------------------------

    xyxy = boxes.xyxy.detach().cpu().numpy()
    confidences = boxes.conf.detach().cpu().numpy()

    # Our model has only one class: person.
    class_ids = boxes.cls.detach().cpu().numpy()

    person_mask = class_ids == 0

    xyxy = xyxy[person_mask]
    confidences = confidences[person_mask]

    person_count = len(xyxy)

    # ---------------------------------------------------------------
    # Basic confidence features
    # ---------------------------------------------------------------

    avg_confidence = float(np.mean(confidences))
    min_confidence = float(np.min(confidences))
    max_confidence = float(np.max(confidences))

    # ---------------------------------------------------------------
    # Bounding-box geometry
    # ---------------------------------------------------------------

    x1 = xyxy[:, 0]
    y1 = xyxy[:, 1]
    x2 = xyxy[:, 2]
    y2 = xyxy[:, 3]

    widths = np.maximum(0, x2 - x1)
    heights = np.maximum(0, y2 - y1)

    areas = widths * heights

    image_area = float(image_width * image_height)

    # Ratio of each bounding box to the complete image.
    bbox_area_ratios = areas / image_area

    total_bbox_area_ratio = float(np.sum(bbox_area_ratios))
    avg_bbox_area_ratio = float(np.mean(bbox_area_ratios))
    max_bbox_area_ratio = float(np.max(bbox_area_ratios))

    # Width and height relative to image dimensions.
    bbox_width_ratios = widths / image_width
    bbox_height_ratios = heights / image_height

    avg_bbox_width_ratio = float(np.mean(bbox_width_ratios))
    avg_bbox_height_ratio = float(np.mean(bbox_height_ratios))

    # ---------------------------------------------------------------
    # People per megapixel
    # ---------------------------------------------------------------

    image_area_megapixels = image_area / 1_000_000.0

    if image_area_megapixels > 0:
        people_per_megapixel = (
            person_count / image_area_megapixels
        )
    else:
        people_per_megapixel = 0.0

    # ---------------------------------------------------------------
    # Bounding-box centers
    # ---------------------------------------------------------------

    centers_x = (x1 + x2) / 2.0
    centers_y = (y1 + y2) / 2.0

    # Normalize center coordinates to 0–1.
    centers_x_norm = centers_x / image_width
    centers_y_norm = centers_y / image_height

    centers = np.column_stack(
        (centers_x_norm, centers_y_norm)
    )

    # ---------------------------------------------------------------
    # Pairwise center distances
    # ---------------------------------------------------------------

    if person_count >= 2:

        # Calculate pairwise normalized distances.
        diff = centers[:, None, :] - centers[None, :, :]

        distance_matrix = np.sqrt(
            np.sum(diff ** 2, axis=2)
        )

        # Only take the upper triangular values.
        upper_indices = np.triu_indices(
            person_count,
            k=1
        )

        pairwise_distances = distance_matrix[
            upper_indices
        ]

        average_center_distance = float(
            np.mean(pairwise_distances)
        )

        # For every person, find the nearest other person.
        nearest_distances = []

        for i in range(person_count):

            distances = distance_matrix[i].copy()

            # Ignore distance to itself.
            distances[i] = np.inf

            nearest_distances.append(
                np.min(distances)
            )

        average_nearest_neighbor_distance = float(
            np.mean(nearest_distances)
        )

    else:

        average_center_distance = 0.0
        average_nearest_neighbor_distance = 0.0

    # ---------------------------------------------------------------
    # 3 × 3 spatial grid
    # ---------------------------------------------------------------

    grid_counts = np.zeros(
        (3, 3),
        dtype=int
    )

    for cx, cy in zip(
        centers_x_norm,
        centers_y_norm
    ):

        # Convert normalized coordinate to grid index.
        col = min(int(cx * 3), 2)
        row = min(int(cy * 3), 2)

        grid_counts[row, col] += 1

    # ---------------------------------------------------------------
    # Return complete feature dictionary
    # ---------------------------------------------------------------

    return {

        # Basic detection
        "person_count": int(person_count),

        "avg_confidence": avg_confidence,
        "min_confidence": min_confidence,
        "max_confidence": max_confidence,

        # Bounding box / occupancy
        "total_bbox_area_ratio": total_bbox_area_ratio,
        "avg_bbox_area_ratio": avg_bbox_area_ratio,
        "max_bbox_area_ratio": max_bbox_area_ratio,

        "avg_bbox_width_ratio": avg_bbox_width_ratio,
        "avg_bbox_height_ratio": avg_bbox_height_ratio,

        # Density
        "people_per_megapixel": float(
            people_per_megapixel
        ),

        # Spatial relationships
        "average_center_distance": (
            average_center_distance
        ),

        "average_nearest_neighbor_distance": (
            average_nearest_neighbor_distance
        ),

        # 3 × 3 spatial grid
        "grid_00": int(grid_counts[0, 0]),
        "grid_01": int(grid_counts[0, 1]),
        "grid_02": int(grid_counts[0, 2]),

        "grid_10": int(grid_counts[1, 0]),
        "grid_11": int(grid_counts[1, 1]),
        "grid_12": int(grid_counts[1, 2]),

        "grid_20": int(grid_counts[2, 0]),
        "grid_21": int(grid_counts[2, 1]),
        "grid_22": int(grid_counts[2, 2]),
    }


print("✅ Feature extraction function created.")

# ---------------------------------------------------------------------
# STEP 6 — Warm up GPU
# ---------------------------------------------------------------------

print("\n[6/9] Warming up GPU...")
print("-" * 80)

warmup_image = str(test_images[0])

print(f"Warm-up image : {Path(warmup_image).name}")
print("Running one inference to initialize CUDA/model kernels...")

_ = model.predict(
    source=warmup_image,
    imgsz=640,
    conf=0.25,
    iou=0.70,
    max_det=500,
    device=DEVICE,
    verbose=False
)

print("✅ GPU/model warm-up complete.")

# ---------------------------------------------------------------------
# STEP 7 — Extract features
# ---------------------------------------------------------------------

print("\n[7/9] Extracting scene-level features...")
print("-" * 80)

print("YOLO configuration:")
print("   Model       : trained YOLO26m")
print("   Image size  : 640")
print("   Confidence  : 0.25")
print("   IoU         : 0.70")
print("   Max det     : 500")
print(f"   Device      : {DEVICE}")

print(f"\nProcessing {len(test_images)} images...")
print("Please wait...\n")

feature_rows = []

start_time = time.perf_counter()

for index, image_path in enumerate(test_images, start=1):

    # ---------------------------------------------------------------
    # Read image dimensions without loading the whole image into
    # memory ourselves.
    # ---------------------------------------------------------------

    from PIL import Image

    with Image.open(image_path) as img:
        width, height = img.size

    # ---------------------------------------------------------------
    # Run YOLO
    # ---------------------------------------------------------------

    results = model.predict(
        source=str(image_path),
        imgsz=640,
        conf=0.25,
        iou=0.70,
        max_det=500,
        device=DEVICE,
        verbose=False
    )

    result = results[0]

    # ---------------------------------------------------------------
    # Extract features
    # ---------------------------------------------------------------

    features = extract_scene_features(
        result=result,
        image_width=width,
        image_height=height
    )

    # Add image metadata.
    features["image"] = image_path.name
    features["image_width"] = width
    features["image_height"] = height

    feature_rows.append(features)

    # ---------------------------------------------------------------
    # Progress information
    # ---------------------------------------------------------------

    if index <= 10 or index % 10 == 0 or index == len(test_images):

        print(
            f"[{index:3d}/{len(test_images)}] "
            f"{image_path.name:<35} "
            f"people={features['person_count']:3d}  "
            f"avg_conf={features['avg_confidence']:.3f}"
        )

elapsed = time.perf_counter() - start_time

print("\n✅ Feature extraction finished.")

# ---------------------------------------------------------------------
# STEP 8 — Create DataFrame and save CSV
# ---------------------------------------------------------------------

print("\n[8/9] Creating feature dataset...")
print("-" * 80)

df = pd.DataFrame(feature_rows)

# Put image information first.
preferred_columns = [
    "image",
    "image_width",
    "image_height",

    "person_count",

    "avg_confidence",
    "min_confidence",
    "max_confidence",

    "total_bbox_area_ratio",
    "avg_bbox_area_ratio",
    "max_bbox_area_ratio",

    "avg_bbox_width_ratio",
    "avg_bbox_height_ratio",

    "people_per_megapixel",

    "average_center_distance",
    "average_nearest_neighbor_distance",

    "grid_00",
    "grid_01",
    "grid_02",
    "grid_10",
    "grid_11",
    "grid_12",
    "grid_20",
    "grid_21",
    "grid_22",
]

df = df[preferred_columns]

OUTPUT_CSV = Path("/content/features_pilot_100.csv")

df.to_csv(
    OUTPUT_CSV,
    index=False
)

print(f"Rows    : {len(df)}")
print(f"Columns : {len(df.columns)}")
print(f"Saved   : {OUTPUT_CSV}")

print(
    f"\nExtraction time : {elapsed:.2f} seconds"
)

print(
    f"Average time/image : "
    f"{elapsed / len(df):.4f} seconds"
)

print(
    f"Approx throughput  : "
    f"{len(df) / elapsed:.2f} images/sec"
)

# ---------------------------------------------------------------------
# STEP 9 — Inspect resulting dataset
# ---------------------------------------------------------------------

print("\n[9/9] Inspecting generated feature dataset...")
print("-" * 80)

print("\nDataset shape:")
print(f"   {df.shape[0]} rows × {df.shape[1]} columns")

print("\nColumn names:")
for i, column in enumerate(df.columns, start=1):
    print(f"   {i:02d}. {column}")

print("\nFirst 5 rows:")
print("-" * 80)

print(
    df.head(5).to_string(
        index=False
    )
)

print("\n\nFeature statistics:")
print("-" * 80)

numeric_columns = df.select_dtypes(
    include=[np.number]
).columns

print(
    df[numeric_columns]
    .describe()
    .round(4)
    .to_string()
)

# ---------------------------------------------------------------------
# Detection summary
# ---------------------------------------------------------------------

print("\n\nDetection summary:")
print("-" * 80)

print(
    f"Images processed       : {len(df)}"
)

print(
    f"Total detected people  : "
    f"{df['person_count'].sum():,}"
)

print(
    f"Average people/image   : "
    f"{df['person_count'].mean():.2f}"
)

print(
    f"Minimum people/image   : "
    f"{df['person_count'].min()}"
)

print(
    f"Maximum people/image   : "
    f"{df['person_count'].max()}"
)

print(
    f"Average confidence     : "
    f"{df['avg_confidence'].mean():.4f}"
)

# ---------------------------------------------------------------------
# Final
# ---------------------------------------------------------------------

print("\n" + "=" * 80)
print(" CELL 4 COMPLETE — PILOT FEATURE EXTRACTION SUCCESSFUL")
print("=" * 80)

print("""
✅ YOLO detections converted into numerical scene-level features.
✅ 100 images processed.
✅ Feature DataFrame created.
✅ CSV file created.

IMPORTANT:
    We have NOT assigned CROWD / NOT-CROWD labels yet.

NEXT:
    We will inspect these features first.
    If everything looks correct, we will run the same pipeline
    over the complete validation dataset.
""")

print(f"📄 Pilot CSV:")
print(f"   {OUTPUT_CSV}")

print("=" * 80)

In [ ]:
# =====================================================================
# CROWD DETECTION PROJECT — V1 ML PHASE
# CELL 5: FULL FEATURE EXTRACTION
# =====================================================================
#
# PURPOSE:
#   Run the trained YOLO26m detector over the COMPLETE CrowdHuman
#   validation set and convert every image into scene-level features.
#
# OUTPUT:
#   /content/crowdhuman_features.csv
#
# IMPORTANT:
#   This cell does NOT train Random Forest.
#   This cell does NOT assign CROWD / NOT-CROWD labels.
#
# Every row = one image
# Every feature = numerical description of that scene
# =====================================================================

from pathlib import Path
import time
import numpy as np
import pandas as pd
import torch
from PIL import Image
from ultralytics import YOLO

print("=" * 85)
print(" CROWD DETECTION PROJECT — V1 ML PHASE")
print(" CELL 5: FULL FEATURE EXTRACTION")
print("=" * 85)

# ---------------------------------------------------------------------
# STEP 1 — Environment
# ---------------------------------------------------------------------

print("\n[1/8] Checking environment...")
print("-" * 85)

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    DEVICE = 0
else:
    print("⚠️ CUDA unavailable. Using CPU.")
    DEVICE = "cpu"

# ---------------------------------------------------------------------
# STEP 2 — Load trained YOLO model
# ---------------------------------------------------------------------

print("\n[2/8] Loading trained YOLO26m model...")
print("-" * 85)

MODEL_PATH = Path("/content/best.pt")

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"❌ Model not found:\n{MODEL_PATH}"
    )

print(f"Model path : {MODEL_PATH}")
print(
    f"Model size : "
    f"{MODEL_PATH.stat().st_size / (1024**2):.2f} MB"
)

model = YOLO(str(MODEL_PATH))

print("\n✅ Model loaded")
print(f"Task   : {model.task}")
print(f"Classes: {model.names}")

# ---------------------------------------------------------------------
# STEP 3 — Locate images
# ---------------------------------------------------------------------

print("\n[3/8] Locating CrowdHuman validation images...")
print("-" * 85)

possible_dirs = [
    Path("/content/CrowdHuman/val_images/Images"),
    Path("/content/crowdhuman_yolo/images/val"),
]

VAL_DIR = None

for directory in possible_dirs:

    print(f"Checking: {directory}")

    if directory.exists():

        count = len(list(directory.glob("*.jpg")))

        if count > 0:
            VAL_DIR = directory
            print(f"   ✅ Found {count:,} images")
            break

if VAL_DIR is None:
    raise FileNotFoundError(
        "❌ CrowdHuman validation images could not be found."
    )

all_images = sorted(VAL_DIR.glob("*.jpg"))

print("\nSelected validation directory:")
print(f"   {VAL_DIR}")

print(f"\nTotal images:")
print(f"   {len(all_images):,}")

if len(all_images) != 4370:
    print(
        f"⚠️ Expected approximately 4,370 images, "
        f"but found {len(all_images):,}."
    )
else:
    print("✅ Expected 4,370 validation images found.")

# ---------------------------------------------------------------------
# STEP 4 — Feature extraction function
# ---------------------------------------------------------------------

print("\n[4/8] Preparing feature extraction function...")
print("-" * 85)

def extract_scene_features(
    result,
    image_width,
    image_height
):
    """
    Convert YOLO person detections for one image into
    scene-level numerical features.
    """

    boxes = result.boxes

    # ---------------------------------------------------------------
    # No detections
    # ---------------------------------------------------------------

    if boxes is None or len(boxes) == 0:

        return {
            "person_count": 0,

            "avg_confidence": 0.0,
            "min_confidence": 0.0,
            "max_confidence": 0.0,

            "total_bbox_area_ratio": 0.0,
            "avg_bbox_area_ratio": 0.0,
            "max_bbox_area_ratio": 0.0,

            "avg_bbox_width_ratio": 0.0,
            "avg_bbox_height_ratio": 0.0,

            "people_per_megapixel": 0.0,

            "average_center_distance": 0.0,
            "average_nearest_neighbor_distance": 0.0,

            "grid_00": 0,
            "grid_01": 0,
            "grid_02": 0,
            "grid_10": 0,
            "grid_11": 0,
            "grid_12": 0,
            "grid_20": 0,
            "grid_21": 0,
            "grid_22": 0,
        }

    # ---------------------------------------------------------------
    # Extract tensors
    # ---------------------------------------------------------------

    xyxy = boxes.xyxy.detach().cpu().numpy()
    confidences = boxes.conf.detach().cpu().numpy()
    class_ids = boxes.cls.detach().cpu().numpy()

    # Class 0 = person.
    person_mask = class_ids == 0

    xyxy = xyxy[person_mask]
    confidences = confidences[person_mask]

    person_count = len(xyxy)

    # ---------------------------------------------------------------
    # Confidence statistics
    # ---------------------------------------------------------------

    avg_confidence = float(np.mean(confidences))
    min_confidence = float(np.min(confidences))
    max_confidence = float(np.max(confidences))

    # ---------------------------------------------------------------
    # Bounding box geometry
    # ---------------------------------------------------------------

    x1 = xyxy[:, 0]
    y1 = xyxy[:, 1]
    x2 = xyxy[:, 2]
    y2 = xyxy[:, 3]

    widths = np.maximum(0, x2 - x1)
    heights = np.maximum(0, y2 - y1)

    areas = widths * heights

    image_area = float(
        image_width * image_height
    )

    bbox_area_ratios = (
        areas / image_area
    )

    total_bbox_area_ratio = float(
        np.sum(bbox_area_ratios)
    )

    avg_bbox_area_ratio = float(
        np.mean(bbox_area_ratios)
    )

    max_bbox_area_ratio = float(
        np.max(bbox_area_ratios)
    )

    avg_bbox_width_ratio = float(
        np.mean(widths / image_width)
    )

    avg_bbox_height_ratio = float(
        np.mean(heights / image_height)
    )

    # ---------------------------------------------------------------
    # Density
    # ---------------------------------------------------------------

    image_area_megapixels = (
        image_area / 1_000_000.0
    )

    people_per_megapixel = (
        person_count / image_area_megapixels
        if image_area_megapixels > 0
        else 0.0
    )

    # ---------------------------------------------------------------
    # Bounding box centers
    # ---------------------------------------------------------------

    centers_x = (
        x1 + x2
    ) / 2.0

    centers_y = (
        y1 + y2
    ) / 2.0

    # Normalize coordinates to [0, 1].
    centers_x_norm = (
        centers_x / image_width
    )

    centers_y_norm = (
        centers_y / image_height
    )

    centers = np.column_stack(
        (
            centers_x_norm,
            centers_y_norm
        )
    )

    # ---------------------------------------------------------------
    # Spatial distances
    # ---------------------------------------------------------------

    if person_count >= 2:

        diff = (
            centers[:, None, :]
            -
            centers[None, :, :]
        )

        distance_matrix = np.sqrt(
            np.sum(
                diff ** 2,
                axis=2
            )
        )

        upper_indices = np.triu_indices(
            person_count,
            k=1
        )

        pairwise_distances = (
            distance_matrix[
                upper_indices
            ]
        )

        average_center_distance = float(
            np.mean(pairwise_distances)
        )

        nearest_distances = []

        for i in range(person_count):

            distances = (
                distance_matrix[i].copy()
            )

            distances[i] = np.inf

            nearest_distances.append(
                np.min(distances)
            )

        average_nearest_neighbor_distance = float(
            np.mean(nearest_distances)
        )

    else:

        average_center_distance = 0.0
        average_nearest_neighbor_distance = 0.0

    # ---------------------------------------------------------------
    # 3 × 3 spatial grid
    # ---------------------------------------------------------------

    grid_counts = np.zeros(
        (3, 3),
        dtype=int
    )

    for cx, cy in zip(
        centers_x_norm,
        centers_y_norm
    ):

        col = min(
            int(cx * 3),
            2
        )

        row = min(
            int(cy * 3),
            2
        )

        grid_counts[row, col] += 1

    # ---------------------------------------------------------------
    # Return features
    # ---------------------------------------------------------------

    return {

        "person_count":
            int(person_count),

        "avg_confidence":
            avg_confidence,

        "min_confidence":
            min_confidence,

        "max_confidence":
            max_confidence,

        "total_bbox_area_ratio":
            total_bbox_area_ratio,

        "avg_bbox_area_ratio":
            avg_bbox_area_ratio,

        "max_bbox_area_ratio":
            max_bbox_area_ratio,

        "avg_bbox_width_ratio":
            avg_bbox_width_ratio,

        "avg_bbox_height_ratio":
            avg_bbox_height_ratio,

        "people_per_megapixel":
            float(people_per_megapixel),

        "average_center_distance":
            average_center_distance,

        "average_nearest_neighbor_distance":
            average_nearest_neighbor_distance,

        "grid_00":
            int(grid_counts[0, 0]),

        "grid_01":
            int(grid_counts[0, 1]),

        "grid_02":
            int(grid_counts[0, 2]),

        "grid_10":
            int(grid_counts[1, 0]),

        "grid_11":
            int(grid_counts[1, 1]),

        "grid_12":
            int(grid_counts[1, 2]),

        "grid_20":
            int(grid_counts[2, 0]),

        "grid_21":
            int(grid_counts[2, 1]),

        "grid_22":
            int(grid_counts[2, 2]),
    }


print("✅ Feature extraction function ready.")

# ---------------------------------------------------------------------
# STEP 5 — GPU warm-up
# ---------------------------------------------------------------------

print("\n[5/8] Warming up GPU...")
print("-" * 85)

warmup_image = str(all_images[0])

print(f"Warm-up image : {all_images[0].name}")

_ = model.predict(
    source=warmup_image,
    imgsz=640,
    conf=0.25,
    iou=0.70,
    max_det=500,
    device=DEVICE,
    verbose=False
)

print("✅ GPU/model warm-up complete.")

# ---------------------------------------------------------------------
# STEP 6 — Process ALL images
# ---------------------------------------------------------------------

print("\n[6/8] Extracting features from ALL validation images...")
print("-" * 85)

print("Configuration:")
print("   Model       : YOLO26m")
print("   Image size  : 640")
print("   Confidence  : 0.25")
print("   IoU         : 0.70")
print("   Max det     : 500")
print(f"   Device      : {DEVICE}")
print(f"   Images      : {len(all_images):,}")

print("\nStarting extraction...")
print("Progress will be printed every 250 images.\n")

feature_rows = []

start_time = time.perf_counter()

for index, image_path in enumerate(
    all_images,
    start=1
):

    # ---------------------------------------------------------------
    # Read image dimensions
    # ---------------------------------------------------------------

    with Image.open(image_path) as img:
        width, height = img.size

    # ---------------------------------------------------------------
    # YOLO inference
    # ---------------------------------------------------------------

    results = model.predict(
        source=str(image_path),
        imgsz=640,
        conf=0.25,
        iou=0.70,
        max_det=500,
        device=DEVICE,
        verbose=False
    )

    result = results[0]

    # ---------------------------------------------------------------
    # Feature extraction
    # ---------------------------------------------------------------

    features = extract_scene_features(
        result=result,
        image_width=width,
        image_height=height
    )

    # Add image metadata.
    features["image"] = image_path.name
    features["image_width"] = width
    features["image_height"] = height

    feature_rows.append(features)

    # ---------------------------------------------------------------
    # Progress output
    # ---------------------------------------------------------------

    if (
        index <= 5
        or index % 250 == 0
        or index == len(all_images)
    ):

        elapsed_now = (
            time.perf_counter()
            - start_time
        )

        rate = (
            index / elapsed_now
            if elapsed_now > 0
            else 0
        )

        remaining = (
            len(all_images) - index
        )

        eta_seconds = (
            remaining / rate
            if rate > 0
            else 0
        )

        print(
            f"[{index:4d}/{len(all_images)}] "
            f"{image_path.name:<35} "
            f"people={features['person_count']:3d}  "
            f"avg_conf={features['avg_confidence']:.3f}  "
            f"speed={rate:.2f} img/s  "
            f"ETA={eta_seconds/60:.1f} min"
        )

# ---------------------------------------------------------------------
# STEP 7 — Build complete DataFrame
# ---------------------------------------------------------------------

print("\n[7/8] Building complete feature dataset...")
print("-" * 85)

df = pd.DataFrame(feature_rows)

preferred_columns = [

    "image",
    "image_width",
    "image_height",

    "person_count",

    "avg_confidence",
    "min_confidence",
    "max_confidence",

    "total_bbox_area_ratio",
    "avg_bbox_area_ratio",
    "max_bbox_area_ratio",

    "avg_bbox_width_ratio",
    "avg_bbox_height_ratio",

    "people_per_megapixel",

    "average_center_distance",
    "average_nearest_neighbor_distance",

    "grid_00",
    "grid_01",
    "grid_02",

    "grid_10",
    "grid_11",
    "grid_12",

    "grid_20",
    "grid_21",
    "grid_22",
]

df = df[preferred_columns]

OUTPUT_CSV = Path(
    "/content/crowdhuman_features.csv"
)

df.to_csv(
    OUTPUT_CSV,
    index=False
)

elapsed_total = (
    time.perf_counter()
    - start_time
)

print("\n✅ CSV created successfully.")

print(f"Rows       : {len(df):,}")
print(f"Columns    : {len(df.columns)}")
print(f"CSV path   : {OUTPUT_CSV}")

print(
    f"Total time : "
    f"{elapsed_total / 60:.2f} minutes"
)

print(
    f"Avg/image  : "
    f"{elapsed_total / len(df):.4f} sec"
)

print(
    f"Throughput : "
    f"{len(df) / elapsed_total:.2f} images/sec"
)

# ---------------------------------------------------------------------
# STEP 8 — Final inspection
# ---------------------------------------------------------------------

print("\n[8/8] Final dataset inspection...")
print("-" * 85)

print("\nDataset shape:")
print(
    f"   {df.shape[0]:,} rows × "
    f"{df.shape[1]} columns"
)

print("\nFirst 5 rows:")
print("-" * 85)

print(
    df.head(5).to_string(
        index=False
    )
)

print("\n\nFeature summary:")
print("-" * 85)

numeric_columns = df.select_dtypes(
    include=[np.number]
).columns

print(
    df[numeric_columns]
    .describe()
    .round(4)
    .to_string()
)

print("\n\nDetection summary:")
print("-" * 85)

print(
    f"Total images          : "
    f"{len(df):,}"
)

print(
    f"Total detected people : "
    f"{df['person_count'].sum():,}"
)

print(
    f"Average people/image  : "
    f"{df['person_count'].mean():.2f}"
)

print(
    f"Minimum people/image  : "
    f"{df['person_count'].min()}"
)

print(
    f"Maximum people/image  : "
    f"{df['person_count'].max()}"
)

print(
    f"Average confidence    : "
    f"{df['avg_confidence'].mean():.4f}"
)

print("\n" + "=" * 85)
print(" CELL 5 COMPLETE — FULL FEATURE EXTRACTION FINISHED")
print("=" * 85)

print("""
✅ All CrowdHuman validation images processed.
✅ YOLO detections converted into scene-level features.
✅ Complete CSV created.
✅ No CROWD / NOT-CROWD labels added yet.
✅ No Random Forest trained yet.

NEXT:
    We will inspect the complete feature distribution and then
    prepare the actual labeled dataset for Random Forest.
""")

print(f"📄 Complete feature dataset:")
print(f"   {OUTPUT_CSV}")

print("=" * 85)

In [ ]:
# =====================================================================
# CROWD DETECTION PROJECT — V1 ML PHASE
# CELL 6: GROUND-TRUTH COUNT vs YOLO FEATURE ANALYSIS
# =====================================================================
#
# PURPOSE:
#   Compare the number of people annotated by CrowdHuman ground truth
#   against the number of people detected by our trained YOLO26m.
#
# WHY ARE WE DOING THIS?
#
#   Our feature CSV contains:
#
#       YOLO detections
#              ↓
#       person_count
#              ↓
#       scene-level features
#
#   But CrowdHuman also provides original human annotations.
#
#   We will use those annotations as an INDEPENDENT reference to
#   understand the quality of our YOLO-derived features.
#
# IMPORTANT:
#   - NO Random Forest here.
#   - NO CROWD / NOT-CROWD labels yet.
#   - NO model training.
#
# OUTPUT:
#   /content/crowdhuman_feature_analysis.csv
# =====================================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

print("=" * 85)
print(" CROWD DETECTION PROJECT — V1 ML PHASE")
print(" CELL 6: GROUND-TRUTH COUNT vs YOLO FEATURE ANALYSIS")
print("=" * 85)

# ---------------------------------------------------------------------
# STEP 1 — Check required files
# ---------------------------------------------------------------------

print("\n[1/8] Checking required files...")
print("-" * 85)

FEATURE_CSV = Path(
    "/content/crowdhuman_features.csv"
)

ANNOTATION_FILE = Path(
    "/content/CrowdHuman/annotation_val.odgt"
)

required_files = [
    FEATURE_CSV,
    ANNOTATION_FILE
]

for file_path in required_files:

    if file_path.exists():

        size_mb = (
            file_path.stat().st_size
            / (1024 ** 2)
        )

        print(
            f"✅ {file_path}\n"
            f"   Size: {size_mb:.2f} MB"
        )

    else:

        print(
            f"❌ Missing:\n"
            f"   {file_path}"
        )

        raise FileNotFoundError(
            f"Required file not found:\n{file_path}"
        )

# ---------------------------------------------------------------------
# STEP 2 — Load feature CSV
# ---------------------------------------------------------------------

print("\n[2/8] Loading YOLO feature dataset...")
print("-" * 85)

df = pd.read_csv(FEATURE_CSV)

print("✅ Feature CSV loaded.")

print(f"Rows    : {len(df):,}")
print(f"Columns : {len(df.columns)}")

print("\nImportant existing features:")

for column in [
    "image",
    "person_count",
    "avg_confidence",
    "people_per_megapixel",
    "average_center_distance",
    "average_nearest_neighbor_distance"
]:

    if column in df.columns:
        print(f"   ✅ {column}")
    else:
        print(f"   ❌ {column} MISSING")

# ---------------------------------------------------------------------
# STEP 3 — Read CrowdHuman validation annotations
# ---------------------------------------------------------------------

print("\n[3/8] Reading CrowdHuman ground-truth annotations...")
print("-" * 85)

print(f"Annotation file:")
print(f"   {ANNOTATION_FILE}")

ground_truth_counts = {}

annotation_entries = 0
total_person_boxes = 0
total_mask_entries = 0

with open(
    ANNOTATION_FILE,
    "r",
    encoding="utf-8"
) as f:

    for line_number, line in enumerate(
        f,
        start=1
    ):

        line = line.strip()

        if not line:
            continue

        record = json.loads(line)

        annotation_entries += 1

        image_id = record["ID"]

        person_count = 0
        mask_count = 0

        for gtbox in record.get(
            "gtboxes",
            []
        ):

            tag = gtbox.get("tag")

            if tag == "person":
                person_count += 1
                total_person_boxes += 1

            elif tag == "mask":
                mask_count += 1
                total_mask_entries += 1

        ground_truth_counts[
            image_id
        ] = {
            "ground_truth_person_count":
                person_count,

            "ground_truth_mask_count":
                mask_count
        }

print("\n✅ Ground-truth annotations loaded.")

print(
    f"Annotation entries : "
    f"{annotation_entries:,}"
)

print(
    f"Ground-truth person boxes : "
    f"{total_person_boxes:,}"
)

print(
    f"Ground-truth mask entries  : "
    f"{total_mask_entries:,}"
)

# ---------------------------------------------------------------------
# STEP 4 — Match image IDs
# ---------------------------------------------------------------------

print("\n[4/8] Matching YOLO feature rows with ground truth...")
print("-" * 85)

def image_id_from_filename(filename):
    """
    CrowdHuman image filename:

        284193,faa9000f2678b5e.jpg

    Ground-truth annotation ID:

        284193,faa9000f2678b5e

    Therefore we simply remove '.jpg'.
    """

    return Path(filename).stem


df["image_id"] = df["image"].apply(
    image_id_from_filename
)

df["ground_truth_person_count"] = (
    df["image_id"]
    .map(
        lambda image_id:
            ground_truth_counts.get(
                image_id,
                {}
            ).get(
                "ground_truth_person_count",
                np.nan
            )
    )
)

df["ground_truth_mask_count"] = (
    df["image_id"]
    .map(
        lambda image_id:
            ground_truth_counts.get(
                image_id,
                {}
            ).get(
                "ground_truth_mask_count",
                np.nan
            )
    )
)

matched = (
    df["ground_truth_person_count"]
    .notna()
)

matched_count = int(
    matched.sum()
)

unmatched_count = int(
    (~matched).sum()
)

print(
    f"YOLO feature rows       : "
    f"{len(df):,}"
)

print(
    f"Matched with ground truth : "
    f"{matched_count:,}"
)

print(
    f"Unmatched                 : "
    f"{unmatched_count:,}"
)

if unmatched_count == 0:

    print(
        "✅ PERFECT MATCH — every feature row "
        "has a ground-truth count."
    )

else:

    print(
        "⚠️ Some images could not be matched."
    )

# ---------------------------------------------------------------------
# STEP 5 — Calculate counting differences
# ---------------------------------------------------------------------

print("\n[5/8] Calculating YOLO vs ground-truth differences...")
print("-" * 85)

# Make sure counts are numeric.

df["ground_truth_person_count"] = pd.to_numeric(
    df["ground_truth_person_count"],
    errors="coerce"
)

df["person_count"] = pd.to_numeric(
    df["person_count"],
    errors="coerce"
)

# Difference:
#
# positive → YOLO detected more people
# negative → YOLO detected fewer people

df["count_difference"] = (
    df["person_count"]
    -
    df["ground_truth_person_count"]
)

df["absolute_count_error"] = (
    df["count_difference"]
    .abs()
)

# Percentage error.
#
# Ground truth should normally be > 0 in CrowdHuman.
# We protect against division by zero anyway.

df["count_error_percent"] = np.where(
    df["ground_truth_person_count"] > 0,

    (
        df["absolute_count_error"]
        /
        df["ground_truth_person_count"]
    )
    * 100,

    0.0
)

# Relative detection ratio:
#
# YOLO count / ground-truth count
#
# 1.0  = perfect count
# <1.0 = under-counting
# >1.0 = over-counting

df["detection_count_ratio"] = np.where(
    df["ground_truth_person_count"] > 0,

    df["person_count"]
    /
    df["ground_truth_person_count"],

    0.0
)

# ---------------------------------------------------------------------
# STEP 6 — Overall counting statistics
# ---------------------------------------------------------------------

print("\n[6/8] Overall counting analysis...")
print("-" * 85)

valid_df = df[
    df["ground_truth_person_count"].notna()
].copy()

gt = valid_df[
    "ground_truth_person_count"
]

pred = valid_df[
    "person_count"
]

absolute_error = valid_df[
    "absolute_count_error"
]

count_difference = valid_df[
    "count_difference"
]

ratio = valid_df[
    "detection_count_ratio"
]

print("\nGround-truth people/image:")
print(
    f"   Mean   : {gt.mean():.2f}"
)

print(
    f"   Median : {gt.median():.2f}"
)

print(
    f"   Min    : {gt.min():.0f}"
)

print(
    f"   Max    : {gt.max():.0f}"
)

print("\nYOLO detected people/image:")
print(
    f"   Mean   : {pred.mean():.2f}"
)

print(
    f"   Median : {pred.median():.2f}"
)

print(
    f"   Min    : {pred.min():.0f}"
)

print(
    f"   Max    : {pred.max():.0f}"
)

print("\nCounting error:")
print(
    f"   Mean absolute error : "
    f"{absolute_error.mean():.2f} people"
)

print(
    f"   Median absolute error : "
    f"{absolute_error.median():.2f} people"
)

print(
    f"   Mean signed difference : "
    f"{count_difference.mean():.2f}"
)

print(
    f"   Mean detection ratio : "
    f"{ratio.mean():.3f}"
)

print(
    f"   Median detection ratio : "
    f"{ratio.median():.3f}"
)

# ---------------------------------------------------------------------
# STEP 7 — Analyze candidate crowd thresholds
# ---------------------------------------------------------------------

print("\n[7/8] Analyzing possible crowd thresholds...")
print("-" * 85)

print("""
IMPORTANT:

We are NOT choosing the final threshold yet.

We are only examining what the dataset looks like under several
possible definitions of "crowd".

For example:

    10+ people
    15+ people
    20+ people
    25+ people
    30+ people
    40+ people
    50+ people

The label will be based on the GROUND-TRUTH person count here,
not the YOLO predicted count.

This keeps the label independent from our YOLO features.
""")

candidate_thresholds = [
    10,
    15,
    20,
    25,
    30,
    40,
    50
]

print(
    f"{'Threshold':>12} "
    f"{'CROWD':>10} "
    f"{'NOT-CROWD':>12} "
    f"{'CROWD %':>10}"
)

print("-" * 50)

for threshold in candidate_thresholds:

    crowd_count = int(
        (
            gt >= threshold
        ).sum()
    )

    not_crowd_count = int(
        (
            gt < threshold
        ).sum()
    )

    crowd_percentage = (
        crowd_count
        /
        len(valid_df)
        *
        100
    )

    print(
        f"{threshold:>12} "
        f"{crowd_count:>10,} "
        f"{not_crowd_count:>12,} "
        f"{crowd_percentage:>9.2f}%"
    )

# ---------------------------------------------------------------------
# Find examples
# ---------------------------------------------------------------------

print("\n\nExamples with LOW ground-truth counts:")
print("-" * 85)

low_examples = (
    valid_df
    .sort_values(
        "ground_truth_person_count"
    )
    .head(5)
)

print(
    low_examples[
        [
            "image",
            "ground_truth_person_count",
            "person_count",
            "count_difference"
        ]
    ]
    .to_string(index=False)
)

print("\n\nExamples with HIGH ground-truth counts:")
print("-" * 85)

high_examples = (
    valid_df
    .sort_values(
        "ground_truth_person_count",
        ascending=False
    )
    .head(5)
)

print(
    high_examples[
        [
            "image",
            "ground_truth_person_count",
            "person_count",
            "count_difference"
        ]
    ]
    .to_string(index=False)
)

# ---------------------------------------------------------------------
# STEP 8 — Save analysis CSV
# ---------------------------------------------------------------------

print("\n[8/8] Saving analysis dataset...")
print("-" * 85)

ANALYSIS_CSV = Path(
    "/content/crowdhuman_feature_analysis.csv"
)

df.to_csv(
    ANALYSIS_CSV,
    index=False
)

print("✅ Analysis CSV saved.")

print(
    f"Path:\n"
    f"   {ANALYSIS_CSV}"
)

print(
    f"Rows:\n"
    f"   {len(df):,}"
)

print(
    f"Columns:\n"
    f"   {len(df.columns)}"
)

print("\n" + "=" * 85)
print(" CELL 6 COMPLETE")
print("=" * 85)

print("""
✅ YOLO features successfully matched with CrowdHuman ground truth.
✅ Ground-truth person counts calculated.
✅ YOLO-vs-ground-truth counting error calculated.
✅ Candidate crowd thresholds analyzed.
✅ Analysis CSV saved.

IMPORTANT:
    We have still NOT created the final CROWD / NOT-CROWD label.

NEXT:
    We will use this analysis to choose a defensible labeling strategy
    before training Random Forest.
""")

print("=" * 85)

In [ ]:
# =====================================================================
# CROWD DETECTION PROJECT — V1 ML PHASE
# CELL 7: CREATE FINAL LABELED ML DATASET
# =====================================================================
#
# PURPOSE:
#   Convert our analyzed feature dataset into the final dataset that
#   will be used by the Random Forest classifier.
#
# V1 CROWD DEFINITION:
#
#       Ground-truth person count >= 20
#                   ↓
#                 CROWD
#
#       Ground-truth person count < 20
#                   ↓
#               NOT-CROWD
#
# IMPORTANT:
#
#   The label is created from CrowdHuman GROUND-TRUTH annotations.
#
#   The Random Forest features will come from YOLO26m detections.
#
#   Therefore:
#
#       GROUND TRUTH → LABEL
#       YOLO          → FEATURES
#
#   This avoids directly creating the label from YOLO's own
#   person_count prediction.
#
# OUTPUT:
#
#   /content/crowdhuman_ml_dataset.csv
#
# NEXT:
#
#   Train/test split
#   Random Forest training
#   Evaluation
#
# =====================================================================

from pathlib import Path
import numpy as np
import pandas as pd

print("=" * 90)
print(" CROWD DETECTION PROJECT — V1 ML PHASE")
print(" CELL 7: CREATE FINAL LABELED ML DATASET")
print("=" * 90)

# ---------------------------------------------------------------------
# STEP 1 — Configuration
# ---------------------------------------------------------------------

print("\n[1/9] Setting V1 labeling configuration...")
print("-" * 90)

INPUT_CSV = Path(
    "/content/crowdhuman_feature_analysis.csv"
)

OUTPUT_CSV = Path(
    "/content/crowdhuman_ml_dataset.csv"
)

# ---------------------------------------------------------------
# V1 operational definition of crowd.
# ---------------------------------------------------------------

CROWD_THRESHOLD = 20

print(
    f"\nV1 CROWD threshold : "
    f"{CROWD_THRESHOLD} people"
)

print("""
V1 labeling rule:

    Ground-truth people >= 20
                ↓
              CROWD

    Ground-truth people < 20
                ↓
           NOT-CROWD

This threshold is being used as an operational definition for
our V1 experiment.
""")

# ---------------------------------------------------------------------
# STEP 2 — Check input file
# ---------------------------------------------------------------------

print("\n[2/9] Checking input analysis dataset...")
print("-" * 90)

if not INPUT_CSV.exists():

    raise FileNotFoundError(
        f"""
❌ Input CSV not found:

{INPUT_CSV}

Make sure Cell 6 completed successfully.
"""
    )

print("✅ Input CSV found.")

print(
    f"File size : "
    f"{INPUT_CSV.stat().st_size / (1024**2):.2f} MB"
)

# ---------------------------------------------------------------------
# STEP 3 — Load dataset
# ---------------------------------------------------------------------

print("\n[3/9] Loading feature + ground-truth dataset...")
print("-" * 90)

df = pd.read_csv(
    INPUT_CSV
)

print("✅ Dataset loaded.")

print(
    f"Rows    : {len(df):,}"
)

print(
    f"Columns : {len(df.columns)}"
)

print("\nRequired columns:")

required_columns = [

    # Image identifier
    "image",

    # YOLO features
    "person_count",
    "avg_confidence",
    "min_confidence",
    "max_confidence",

    "total_bbox_area_ratio",
    "avg_bbox_area_ratio",
    "max_bbox_area_ratio",

    "avg_bbox_width_ratio",
    "avg_bbox_height_ratio",

    "people_per_megapixel",

    "average_center_distance",
    "average_nearest_neighbor_distance",

    # Ground truth
    "ground_truth_person_count"
]

missing_columns = []

for column in required_columns:

    if column in df.columns:

        print(
            f"   ✅ {column}"
        )

    else:

        print(
            f"   ❌ {column} MISSING"
        )

        missing_columns.append(
            column
        )

if missing_columns:

    raise ValueError(
        "Missing required columns:\n"
        +
        "\n".join(
            missing_columns
        )
    )

# ---------------------------------------------------------------------
# STEP 4 — Validate ground-truth counts
# ---------------------------------------------------------------------

print("\n[4/9] Validating ground-truth counts...")
print("-" * 90)

gt_counts = pd.to_numeric(
    df["ground_truth_person_count"],
    errors="coerce"
)

missing_gt = int(
    gt_counts.isna().sum()
)

negative_gt = int(
    (gt_counts < 0).sum()
)

zero_gt = int(
    (gt_counts == 0).sum()
)

print(
    f"Total images             : "
    f"{len(df):,}"
)

print(
    f"Missing ground-truth     : "
    f"{missing_gt}"
)

print(
    f"Negative ground-truth    : "
    f"{negative_gt}"
)

print(
    f"Zero ground-truth counts : "
    f"{zero_gt}"
)

if missing_gt > 0:

    raise ValueError(
        "❌ Missing ground-truth counts detected."
    )

if negative_gt > 0:

    raise ValueError(
        "❌ Negative ground-truth counts detected."
    )

print(
    "✅ Ground-truth counts are valid."
)

# ---------------------------------------------------------------------
# STEP 5 — Create labels
# ---------------------------------------------------------------------

print("\n[5/9] Creating CROWD / NOT-CROWD labels...")
print("-" * 90)

# ---------------------------------------------------------------
# Numeric label:
#
#     0 = NOT-CROWD
#     1 = CROWD
# ---------------------------------------------------------------

df["crowd_label"] = np.where(
    df["ground_truth_person_count"]
    >= CROWD_THRESHOLD,
    1,
    0
)

# ---------------------------------------------------------------
# Human-readable label.
# ---------------------------------------------------------------

df["crowd_status"] = np.where(
    df["crowd_label"] == 1,
    "CROWD",
    "NOT-CROWD"
)

print("✅ Labels created.")

# ---------------------------------------------------------------------
# STEP 6 — Class distribution
# ---------------------------------------------------------------------

print("\n[6/9] Checking class distribution...")
print("-" * 90)

class_counts = (
    df["crowd_status"]
    .value_counts()
)

crowd_count = int(
    (
        df["crowd_label"] == 1
    ).sum()
)

not_crowd_count = int(
    (
        df["crowd_label"] == 0
    ).sum()
)

total_images = len(df)

crowd_percentage = (
    crowd_count
    /
    total_images
    *
    100
)

not_crowd_percentage = (
    not_crowd_count
    /
    total_images
    *
    100
)

print(
    f"\nTotal images : "
    f"{total_images:,}"
)

print(
    "\nCROWD:"
)

print(
    f"   Images      : "
    f"{crowd_count:,}"
)

print(
    f"   Percentage  : "
    f"{crowd_percentage:.2f}%"
)

print(
    "\nNOT-CROWD:"
)

print(
    f"   Images      : "
    f"{not_crowd_count:,}"
)

print(
    f"   Percentage  : "
    f"{not_crowd_percentage:.2f}%"
)

print("\nClass balance:")
print(
    f"   CROWD     : "
    f"{crowd_percentage:.2f}%"
)

print(
    f"   NOT-CROWD : "
    f"{not_crowd_percentage:.2f}%"
)

# ---------------------------------------------------------------------
# STEP 7 — Analyze features by class
# ---------------------------------------------------------------------

print("\n[7/9] Comparing feature distributions between classes...")
print("-" * 90)

print("""
This section is important.

Before training Random Forest, we want to see whether our
YOLO-derived features actually differ between CROWD and NOT-CROWD.

For example:

    CROWD scenes should generally have:
        • more detected people
        • greater people-per-area density
        • smaller average distances
        • different spatial distributions

If the features show meaningful differences, that is a good
indication that Random Forest has useful information to learn.
""")

comparison_columns = [

    "person_count",

    "avg_confidence",

    "total_bbox_area_ratio",

    "avg_bbox_area_ratio",

    "people_per_megapixel",

    "average_center_distance",

    "average_nearest_neighbor_distance"
]

comparison = (
    df.groupby(
        "crowd_status"
    )[comparison_columns]
    .agg(
        ["mean", "median"]
    )
)

print(
    comparison.round(4).to_string()
)

# ---------------------------------------------------------------------
# Count agreement analysis
# ---------------------------------------------------------------------

print("\n\nGround-truth vs YOLO count by class:")
print("-" * 90)

for status in [
    "NOT-CROWD",
    "CROWD"
]:

    subset = df[
        df["crowd_status"] == status
    ]

    print(
        f"\n{status}"
    )

    print(
        f"   Images : "
        f"{len(subset):,}"
    )

    print(
        f"   Mean ground-truth count : "
        f"{subset['ground_truth_person_count'].mean():.2f}"
    )

    print(
        f"   Mean YOLO count         : "
        f"{subset['person_count'].mean():.2f}"
    )

    print(
        f"   Median ground-truth     : "
        f"{subset['ground_truth_person_count'].median():.0f}"
    )

    print(
        f"   Median YOLO count       : "
        f"{subset['person_count'].median():.0f}"
    )

# ---------------------------------------------------------------------
# STEP 8 — Dataset integrity checks
# ---------------------------------------------------------------------

print("\n[8/9] Running final dataset integrity checks...")
print("-" * 90)

print("\nChecking for missing values...")

missing_values = (
    df.isna()
    .sum()
)

total_missing = int(
    missing_values.sum()
)

print(
    f"Total missing values : "
    f"{total_missing}"
)

if total_missing == 0:

    print(
        "✅ No missing values."
    )

else:

    print(
        "\n⚠️ Missing values found:"
    )

    print(
        missing_values[
            missing_values > 0
        ]
        .to_string()
    )

print("\nChecking labels...")

unique_labels = sorted(
    df["crowd_label"]
    .unique()
    .tolist()
)

print(
    f"Unique numeric labels : "
    f"{unique_labels}"
)

if unique_labels == [0, 1]:

    print(
        "✅ Both classes are present."
    )

else:

    raise ValueError(
        "❌ Dataset does not contain both classes."
    )

print("\nChecking label consistency...")

label_mismatch = (
    (
        (
            df["ground_truth_person_count"]
            >= CROWD_THRESHOLD
        ).astype(int)
        !=
        df["crowd_label"]
    )
    .sum()
)

print(
    f"Label mismatches : "
    f"{label_mismatch}"
)

if label_mismatch == 0:

    print(
        "✅ Label consistency check passed."
    )

else:

    raise ValueError(
        "❌ Label consistency check failed."
    )

# ---------------------------------------------------------------------
# STEP 9 — Save final ML dataset
# ---------------------------------------------------------------------

print("\n[9/9] Saving final ML dataset...")
print("-" * 90)

# ---------------------------------------------------------------
# Put important columns first.
# ---------------------------------------------------------------

first_columns = [

    "image",

    # Ground truth used to create label
    "ground_truth_person_count",

    # Label
    "crowd_label",
    "crowd_status",

    # YOLO-derived features
    "person_count",
    "avg_confidence",
    "min_confidence",
    "max_confidence",

    "total_bbox_area_ratio",
    "avg_bbox_area_ratio",
    "max_bbox_area_ratio",

    "avg_bbox_width_ratio",
    "avg_bbox_height_ratio",

    "people_per_megapixel",

    "average_center_distance",
    "average_nearest_neighbor_distance",

    # Spatial grid
    "grid_00",
    "grid_01",
    "grid_02",

    "grid_10",
    "grid_11",
    "grid_12",

    "grid_20",
    "grid_21",
    "grid_22",
]

# Keep image dimensions and other metadata after the main columns.
remaining_columns = [
    column
    for column in df.columns
    if column not in first_columns
]

df = df[
    first_columns
    +
    remaining_columns
]

df.to_csv(
    OUTPUT_CSV,
    index=False
)

print(
    "✅ Final ML dataset saved."
)

print(
    f"\nOutput file:"
)

print(
    f"   {OUTPUT_CSV}"
)

print(
    f"\nFile size:"
)

print(
    f"   "
    f"{OUTPUT_CSV.stat().st_size / (1024**2):.2f} MB"
)

print(
    f"\nRows:"
)

print(
    f"   {len(df):,}"
)

print(
    f"\nColumns:"
)

print(
    f"   {len(df.columns)}"
)

# ---------------------------------------------------------------------
# Final preview
# ---------------------------------------------------------------------

print("\n\nFINAL DATASET PREVIEW")
print("=" * 90)

preview_columns = [

    "image",
    "ground_truth_person_count",
    "crowd_label",
    "crowd_status",
    "person_count",
    "avg_confidence",
    "people_per_megapixel",
    "average_nearest_neighbor_distance"
]

print(
    df[
        preview_columns
    ]
    .head(10)
    .to_string(index=False)
)

# ---------------------------------------------------------------------
# Final summary
# ---------------------------------------------------------------------

print("\n" + "=" * 90)
print(" CELL 7 COMPLETE — FINAL LABELED ML DATASET CREATED")
print("=" * 90)

print("""
✅ 4,370 images processed.
✅ Ground-truth person counts verified.
✅ V1 CROWD definition applied.
✅ CROWD / NOT-CROWD labels created.
✅ Class distribution checked.
✅ YOLO features compared between classes.
✅ Missing-value checks passed.
✅ Label consistency verified.
✅ Final ML CSV saved.

V1 LABEL DEFINITION:

    Ground-truth people >= 20  →  CROWD
    Ground-truth people <  20  →  NOT-CROWD

DATA FLOW:

    CrowdHuman ground truth
             │
             └──────→ CROWD / NOT-CROWD LABEL
                              │
                              │
    YOLO26m ──→ FEATURES ────┘
                              │
                              ▼
                       RANDOM FOREST

NEXT PHASE:
    Train/Test split
          ↓
    Random Forest training
          ↓
    Evaluation
          ↓
    Confusion Matrix
          ↓
    Precision / Recall / F1
          ↓
    ROC-AUC
""")

print(
    f"📄 Final ML dataset:"
)

print(
    f"   {OUTPUT_CSV}"
)

print("=" * 90)

In [ ]:
# =============================================================================
# CROWD DETECTION PROJECT — V1 ML PHASE
# CELL 8: TRAIN / TEST SPLIT
# =============================================================================
#
# Purpose:
#   Prepare the final dataset for Random Forest training.
#
# Dataset:
#   crowdhuman_ml_dataset.csv
#
# Target:
#   crowd_label
#       0 = NOT-CROWD
#       1 = CROWD
#
# Split:
#   80% → Training
#   20% → Testing
#
# Important:
#   We use STRATIFIED splitting so that the CROWD / NOT-CROWD ratio
#   remains approximately the same in both sets.
#
# =============================================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

print("=" * 90)
print(" CROWD DETECTION PROJECT — V1 ML PHASE")
print(" CELL 8: TRAIN / TEST SPLIT")
print("=" * 90)


# =============================================================================
# [1/8] Configuration
# =============================================================================

DATASET_PATH = "/content/crowdhuman_ml_dataset.csv"

TEST_SIZE = 0.20
RANDOM_STATE = 42

print("\n[1/8] Configuration")
print("-" * 90)

print(f"Dataset       : {DATASET_PATH}")
print(f"Test size     : {TEST_SIZE * 100:.0f}%")
print(f"Train size    : {(1 - TEST_SIZE) * 100:.0f}%")
print(f"Random state  : {RANDOM_STATE}")
print("Split method  : Stratified")


# =============================================================================
# [2/8] Load dataset
# =============================================================================

print("\n[2/8] Loading final ML dataset...")
print("-" * 90)

df = pd.read_csv(DATASET_PATH)

print("✅ Dataset loaded successfully.")

print(f"\nRows    : {len(df):,}")
print(f"Columns : {len(df.columns)}")


# =============================================================================
# [3/8] Define target and feature columns
# =============================================================================

print("\n[3/8] Defining ML features and target...")
print("-" * 90)

# These are the features generated from YOLO detections.
FEATURE_COLUMNS = [
    "person_count",
    "avg_confidence",
    "min_confidence",
    "max_confidence",

    "total_bbox_area_ratio",
    "avg_bbox_area_ratio",
    "max_bbox_area_ratio",

    "avg_bbox_width_ratio",
    "avg_bbox_height_ratio",

    "people_per_megapixel",

    "average_center_distance",
    "average_nearest_neighbor_distance",

    # 3 × 3 spatial grid features
    "grid_00",
    "grid_01",
    "grid_02",
    "grid_10",
    "grid_11",
    "grid_12",
    "grid_20",
    "grid_21",
    "grid_22",
]

TARGET_COLUMN = "crowd_label"


# Check that every required column exists.

missing_features = [
    col for col in FEATURE_COLUMNS
    if col not in df.columns
]

if missing_features:
    raise ValueError(
        f"❌ Missing feature columns: {missing_features}"
    )

if TARGET_COLUMN not in df.columns:
    raise ValueError(
        f"❌ Target column '{TARGET_COLUMN}' not found!"
    )

print(f"Number of ML features : {len(FEATURE_COLUMNS)}")
print(f"Target column         : {TARGET_COLUMN}")

print("\nFeatures:")
for i, feature in enumerate(FEATURE_COLUMNS, start=1):
    print(f"   {i:2d}. {feature}")


# =============================================================================
# [4/8] Create X and y
# =============================================================================

print("\n[4/8] Creating feature matrix X and target y...")
print("-" * 90)

X = df[FEATURE_COLUMNS].copy()
y = df[TARGET_COLUMN].copy()

print("✅ X and y created.")

print(f"\nX shape : {X.shape}")
print(f"y shape : {y.shape}")

print("\nTarget values:")
print(y.value_counts().sort_index())


# =============================================================================
# [5/8] Check data before splitting
# =============================================================================

print("\n[5/8] Checking dataset before split...")
print("-" * 90)

# Missing values
missing_values = X.isnull().sum().sum()

print(f"Missing feature values : {missing_values}")

if missing_values != 0:
    raise ValueError("❌ Missing feature values detected!")

# Infinite values
infinite_values = np.isinf(X.to_numpy()).sum()

print(f"Infinite feature values : {infinite_values}")

if infinite_values != 0:
    raise ValueError("❌ Infinite feature values detected!")

# Target values
unique_targets = sorted(y.unique())

print(f"Unique target values    : {unique_targets}")

if unique_targets != [0, 1]:
    raise ValueError(
        f"❌ Expected target values [0, 1], found {unique_targets}"
    )

print("✅ Pre-split integrity checks passed.")


# =============================================================================
# [6/8] Perform STRATIFIED train/test split
# =============================================================================

print("\n[6/8] Performing stratified train/test split...")
print("-" * 90)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("✅ Train/test split completed.")


# =============================================================================
# [7/8] Verify class distribution
# =============================================================================

print("\n[7/8] Verifying class distribution...")
print("-" * 90)

def show_distribution(name, labels):
    counts = labels.value_counts().sort_index()
    percentages = labels.value_counts(normalize=True).sort_index() * 100

    print(f"\n{name}")
    print("-" * 50)

    print(
        f"NOT-CROWD : "
        f"{counts.get(0, 0):4d} images "
        f"({percentages.get(0, 0):6.2f}%)"
    )

    print(
        f"CROWD     : "
        f"{counts.get(1, 0):4d} images "
        f"({percentages.get(1, 0):6.2f}%)"
    )


show_distribution("FULL DATASET", y)
show_distribution("TRAINING SET", y_train)
show_distribution("TEST SET", y_test)


# =============================================================================
# [8/8] Final split summary
# =============================================================================

print("\n[8/8] Final split summary")
print("-" * 90)

print(f"Total samples : {len(df):,}")
print(f"Training      : {len(X_train):,}")
print(f"Testing       : {len(X_test):,}")

print(f"\nTraining features : {X_train.shape[1]}")
print(f"Testing features  : {X_test.shape[1]}")

print("\nFeature matrix:")
print(f"   X_train : {X_train.shape}")
print(f"   X_test  : {X_test.shape}")

print("\nTarget vector:")
print(f"   y_train : {y_train.shape}")
print(f"   y_test  : {y_test.shape}")

print("\n" + "=" * 90)
print(" CELL 8 COMPLETE")
print("=" * 90)

print("""
✅ Dataset loaded.
✅ 21 YOLO-derived ML features selected.
✅ Target = crowd_label.
✅ Missing-value check passed.
✅ Infinite-value check passed.
✅ Target-label check passed.
✅ Stratified 80/20 split completed.
✅ Class distribution verified.

NEXT:
    Cell 9 → Train Random Forest
""")

print("=" * 90)

In [ ]:
# =============================================================================
# CROWD DETECTION PROJECT — V1 ML PHASE
# CELL 9: RANDOM FOREST TRAINING
# =============================================================================
#
# Purpose:
#   Train the V1 scene-level CROWD / NOT-CROWD classifier.
#
# Pipeline:
#
#   YOLO26m
#       ↓
#   21 scene-level features
#       ↓
#   Random Forest
#       ↓
#   CROWD / NOT-CROWD
#
# Target:
#   0 = NOT-CROWD
#   1 = CROWD
#
# Important:
#   The test set is NOT used during training.
#
# =============================================================================

import pandas as pd
import numpy as np
import pickle
import time

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

print("=" * 90)
print(" CROWD DETECTION PROJECT — V1 ML PHASE")
print(" CELL 9: RANDOM FOREST TRAINING")
print("=" * 90)


# =============================================================================
# [1/8] Configuration
# =============================================================================

print("\n[1/8] Random Forest configuration")
print("-" * 90)

RANDOM_STATE = 42

N_ESTIMATORS = 300
MAX_DEPTH = None
MIN_SAMPLES_SPLIT = 2
MIN_SAMPLES_LEAF = 1
MAX_FEATURES = "sqrt"

MODEL_PATH = "/content/crowd_random_forest_v1.pkl"

print(f"Number of trees       : {N_ESTIMATORS}")
print(f"Maximum depth         : {MAX_DEPTH}")
print(f"Minimum samples split : {MIN_SAMPLES_SPLIT}")
print(f"Minimum samples leaf  : {MIN_SAMPLES_LEAF}")
print(f"Max features          : {MAX_FEATURES}")
print(f"Random state          : {RANDOM_STATE}")


# =============================================================================
# [2/8] Verify training/test data exists
# =============================================================================

print("\n[2/8] Verifying train/test data...")
print("-" * 90)

required_variables = [
    "X_train",
    "X_test",
    "y_train",
    "y_test"
]

for variable in required_variables:
    if variable not in globals():
        raise RuntimeError(
            f"❌ {variable} is not available. "
            "Please run Cell 8 first."
        )

print("✅ X_train found")
print("✅ X_test found")
print("✅ y_train found")
print("✅ y_test found")

print(f"\nX_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}")
print(f"y_test  : {y_test.shape}")


# =============================================================================
# [3/8] Display training class distribution
# =============================================================================

print("\n[3/8] Training class distribution")
print("-" * 90)

train_counts = y_train.value_counts().sort_index()

print(
    f"NOT-CROWD : "
    f"{train_counts[0]:4d} samples "
    f"({train_counts[0] / len(y_train) * 100:.2f}%)"
)

print(
    f"CROWD     : "
    f"{train_counts[1]:4d} samples "
    f"({train_counts[1] / len(y_train) * 100:.2f}%)"
)


# =============================================================================
# [4/8] Create Random Forest
# =============================================================================

print("\n[4/8] Creating Random Forest classifier...")
print("-" * 90)

rf_model = RandomForestClassifier(
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
    min_samples_split=MIN_SAMPLES_SPLIT,
    min_samples_leaf=MIN_SAMPLES_LEAF,
    max_features=MAX_FEATURES,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    class_weight=None
)

print("✅ Random Forest created.")
print(f"Trees : {N_ESTIMATORS}")


# =============================================================================
# [5/8] Train model
# =============================================================================

print("\n[5/8] Training Random Forest...")
print("-" * 90)

print("⏳ Training started...")

start_time = time.time()

rf_model.fit(X_train, y_train)

training_time = time.time() - start_time

print("✅ Training completed.")

print(f"Training time : {training_time:.2f} seconds")


# =============================================================================
# [6/8] Generate predictions
# =============================================================================

print("\n[6/8] Generating predictions...")
print("-" * 90)

# Training predictions
y_train_pred = rf_model.predict(X_train)

# Completely unseen test-set predictions
y_test_pred = rf_model.predict(X_test)

# Probability of CROWD
y_test_proba = rf_model.predict_proba(X_test)[:, 1]

print("✅ Predictions generated.")

print(f"Training predictions : {len(y_train_pred):,}")
print(f"Test predictions     : {len(y_test_pred):,}")


# =============================================================================
# [7/8] Basic performance check
# =============================================================================

print("\n[7/8] Basic model performance")
print("-" * 90)

train_accuracy = accuracy_score(
    y_train,
    y_train_pred
)

test_accuracy = accuracy_score(
    y_test,
    y_test_pred
)

print(f"Training accuracy : {train_accuracy:.4f}")
print(f"Training accuracy : {train_accuracy * 100:.2f}%")

print()

print(f"Test accuracy     : {test_accuracy:.4f}")
print(f"Test accuracy     : {test_accuracy * 100:.2f}%")

print("\nImportant:")
print("Training accuracy shows how well the forest fits the training data.")
print("Test accuracy is the important first estimate of generalization.")


# =============================================================================
# [8/8] Feature importance + model saving
# =============================================================================

print("\n[8/8] Feature importance and model saving")
print("-" * 90)

# -------------------------------------------------------------------------
# Feature importance
# -------------------------------------------------------------------------

importance_df = pd.DataFrame({
    "feature": X_train.columns,
    "importance": rf_model.feature_importances_
})

importance_df = importance_df.sort_values(
    by="importance",
    ascending=False
).reset_index(drop=True)

print("\nTop 10 most important features:")
print("-" * 60)

for i, row in importance_df.head(10).iterrows():
    print(
        f"{i + 1:2d}. "
        f"{row['feature']:<40} "
        f"{row['importance']:.6f}"
    )


# -------------------------------------------------------------------------
# Save model
# -------------------------------------------------------------------------

print("\nSaving trained model...")

with open(MODEL_PATH, "wb") as file:
    pickle.dump(rf_model, file)

print(f"✅ Model saved:")
print(f"   {MODEL_PATH}")


# -------------------------------------------------------------------------
# Final summary
# -------------------------------------------------------------------------

print("\n" + "=" * 90)
print(" CELL 9 COMPLETE")
print("=" * 90)

print(f"""
✅ Random Forest trained successfully.
✅ {N_ESTIMATORS} decision trees created.
✅ Training samples : {len(X_train):,}
✅ Test samples     : {len(X_test):,}
✅ Features         : {X_train.shape[1]}

TRAINING ACCURACY:
    {train_accuracy * 100:.2f}%

TEST ACCURACY:
    {test_accuracy * 100:.2f}%

MODEL:
    {MODEL_PATH}

NEXT:
    Cell 10 → Detailed evaluation
                 • Confusion Matrix
                 • Precision
                 • Recall
                 • F1-score
                 • ROC-AUC
""")

print("=" * 90)

In [ ]:
# =============================================================================
# CROWD DETECTION PROJECT — V1 ML PHASE
# CELL 10: DETAILED RANDOM FOREST EVALUATION
# =============================================================================
#
# Evaluation metrics:
#
#   1. Confusion Matrix
#   2. Accuracy
#   3. Precision
#   4. Recall
#   5. F1-score
#   6. ROC-AUC
#   7. Classification Report
#
# We evaluate ONLY on the unseen test set.
#
# Target:
#   0 = NOT-CROWD
#   1 = CROWD
#
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    roc_curve
)

print("=" * 90)
print(" CROWD DETECTION PROJECT — V1 ML PHASE")
print(" CELL 10: DETAILED RANDOM FOREST EVALUATION")
print("=" * 90)


# =============================================================================
# [1/7] Verify required variables
# =============================================================================

print("\n[1/7] Verifying evaluation data...")
print("-" * 90)

required_variables = [
    "rf_model",
    "X_test",
    "y_test",
    "y_test_pred",
    "y_test_proba"
]

for variable in required_variables:
    if variable not in globals():
        raise RuntimeError(
            f"❌ {variable} is not available. "
            "Please run Cell 9 first."
        )
    print(f"✅ {variable} found")

print(f"\nTest samples : {len(y_test):,}")


# =============================================================================
# [2/7] Confusion Matrix
# =============================================================================

print("\n[2/7] Computing confusion matrix...")
print("-" * 90)

cm = confusion_matrix(
    y_test,
    y_test_pred
)

tn, fp, fn, tp = cm.ravel()

print("\nConfusion Matrix:")
print()
print("                    Predicted")
print("                 NOT-CROWD   CROWD")
print(f"Actual NOT-CROWD    {tn:4d}       {fp:4d}")
print(f"Actual CROWD        {fn:4d}       {tp:4d}")

print("\nInterpretation:")
print(f"   True Negatives  (TN) : {tn}")
print(f"   False Positives (FP) : {fp}")
print(f"   False Negatives (FN) : {fn}")
print(f"   True Positives  (TP) : {tp}")


# =============================================================================
# [3/7] Calculate classification metrics
# =============================================================================

print("\n[3/7] Calculating classification metrics...")
print("-" * 90)

accuracy = accuracy_score(
    y_test,
    y_test_pred
)

precision = precision_score(
    y_test,
    y_test_pred,
    zero_division=0
)

recall = recall_score(
    y_test,
    y_test_pred,
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_test_pred,
    zero_division=0
)

roc_auc = roc_auc_score(
    y_test,
    y_test_proba
)

print(f"Accuracy  : {accuracy:.4f}  ({accuracy * 100:.2f}%)")
print(f"Precision : {precision:.4f}  ({precision * 100:.2f}%)")
print(f"Recall    : {recall:.4f}  ({recall * 100:.2f}%)")
print(f"F1-score  : {f1:.4f}  ({f1 * 100:.2f}%)")
print(f"ROC-AUC   : {roc_auc:.4f}")


# =============================================================================
# [4/7] Classification report
# =============================================================================

print("\n[4/7] Classification report...")
print("-" * 90)

report = classification_report(
    y_test,
    y_test_pred,
    target_names=[
        "NOT-CROWD",
        "CROWD"
    ],
    digits=4,
    zero_division=0
)

print(report)


# =============================================================================
# [5/7] Interpret CROWD detection performance
# =============================================================================

print("\n[5/7] CROWD detection interpretation")
print("-" * 90)

print(f"""
For the CROWD class:

Precision = {precision * 100:.2f}%
    Of all images predicted as CROWD,
    {precision * 100:.2f}% were actually CROWD.

Recall = {recall * 100:.2f}%
    Of all actual CROWD images,
    the model detected {recall * 100:.2f}% of them.

F1-score = {f1 * 100:.2f}%
    Harmonic mean of precision and recall.

ROC-AUC = {roc_auc:.4f}
    Measures how well the model separates CROWD
    from NOT-CROWD across probability thresholds.
""")


# =============================================================================
# [6/7] Plot confusion matrix
# =============================================================================

print("\n[6/7] Generating confusion matrix visualization...")
print("-" * 90)

fig, ax = plt.subplots(figsize=(7, 6))

display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["NOT-CROWD", "CROWD"]
)

display.plot(
    ax=ax,
    values_format="d"
)

ax.set_title("Random Forest V1 — Confusion Matrix")
plt.tight_layout()

plt.show()


# =============================================================================
# [7/7] ROC Curve
# =============================================================================

print("\n[7/7] Generating ROC curve...")
print("-" * 90)

fpr, tpr, thresholds = roc_curve(
    y_test,
    y_test_proba
)

fig, ax = plt.subplots(figsize=(7, 6))

ax.plot(
    fpr,
    tpr,
    label=f"Random Forest (AUC = {roc_auc:.4f})"
)

ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random classifier"
)

ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Random Forest V1 — ROC Curve")

ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


# =============================================================================
# FINAL SUMMARY
# =============================================================================

print("\n" + "=" * 90)
print(" CELL 10 COMPLETE")
print("=" * 90)

print(f"""
FINAL V1 RANDOM FOREST TEST RESULTS
-----------------------------------

Test samples : {len(y_test):,}

Accuracy     : {accuracy * 100:.2f}%
Precision    : {precision * 100:.2f}%
Recall       : {recall * 100:.2f}%
F1-score     : {f1 * 100:.2f}%
ROC-AUC      : {roc_auc:.4f}

Confusion Matrix:
    TN = {tn}
    FP = {fp}
    FN = {fn}
    TP = {tp}

MODEL:
    Random Forest
    300 trees

TARGET:
    0 = NOT-CROWD
    1 = CROWD

NEXT:
    After reviewing these results,
    we will test the complete YOLO26m → Random Forest pipeline
    on individual images.
""")

print("=" * 90)

In [ ]:
# =============================================================================
# CROWD DETECTION PROJECT — V1 ML PHASE
# CELL 11: END-TO-END YOLO26m → RANDOM FOREST PREDICTION
# =============================================================================
#
# Purpose:
#   Test the complete V1 crowd-detection pipeline on individual images.
#
# Pipeline:
#
#       INPUT IMAGE
#            ↓
#         YOLO26m
#            ↓
#     Person detections
#            ↓
#     Feature extraction
#            ↓
#      Random Forest
#            ↓
#    CROWD / NOT-CROWD
#
# This cell demonstrates how the trained models work together.
# =============================================================================

from ultralytics import YOLO
import pandas as pd
import numpy as np
import pickle
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as patches


print("=" * 90)
print(" CROWD DETECTION PROJECT — V1 ML PHASE")
print(" CELL 11: END-TO-END YOLO26m → RANDOM FOREST")
print("=" * 90)


# =============================================================================
# [1/9] Configuration
# =============================================================================

print("\n[1/9] Setting pipeline configuration...")
print("-" * 90)

# YOLO_MODEL_PATH = "/content/runs/crowdhuman/yolo26m_v1/weights/best.pt"
YOLO_MODEL_PATH = "/content/best.pt"
RF_MODEL_PATH = "/content/crowd_random_forest_v1.pkl"

IMAGE_DIR = "/content/CrowdHuman/val_images/Images"

CONF_THRESHOLD = 0.25
IOU_THRESHOLD = 0.70
MAX_DETECTIONS = 500

RANDOM_STATE = 42

print(f"YOLO model       : {YOLO_MODEL_PATH}")
print(f"Random Forest    : {RF_MODEL_PATH}")
print(f"Image directory  : {IMAGE_DIR}")
print(f"Confidence       : {CONF_THRESHOLD}")
print(f"IoU threshold    : {IOU_THRESHOLD}")
print(f"Maximum boxes    : {MAX_DETECTIONS}")


# =============================================================================
# [2/9] Verify required files
# =============================================================================

print("\n[2/9] Checking required files...")
print("-" * 90)

required_files = [
    YOLO_MODEL_PATH,
    RF_MODEL_PATH,
    IMAGE_DIR
]

for path in required_files:

    if not Path(path).exists():
        raise FileNotFoundError(
            f"❌ Required path not found:\n{path}"
        )

    print(f"✅ Found: {path}")


# =============================================================================
# [3/9] Load models
# =============================================================================

print("\n[3/9] Loading trained models...")
print("-" * 90)

print("Loading YOLO26m...")

yolo_model = YOLO(YOLO_MODEL_PATH)

print("✅ YOLO26m loaded.")

print("\nLoading Random Forest...")

with open(RF_MODEL_PATH, "rb") as file:
    rf_model_v1 = pickle.load(file)

print("✅ Random Forest loaded.")

print("\nModels ready.")


# =============================================================================
# [4/9] Select test images
# =============================================================================

print("\n[4/9] Selecting test images...")
print("-" * 90)

image_dir = Path(IMAGE_DIR)

image_files = sorted(
    list(image_dir.glob("*.jpg")) +
    list(image_dir.glob("*.jpeg")) +
    list(image_dir.glob("*.png"))
)

if len(image_files) == 0:
    raise RuntimeError(
        "❌ No images found in validation image directory."
    )

print(f"Available images : {len(image_files):,}")

# Select 5 deterministic random images.
random.seed(RANDOM_STATE)

selected_images = random.sample(
    image_files,
    min(5, len(image_files))
)

print("\nSelected images:")

for i, image_path in enumerate(selected_images, start=1):
    print(f"   {i}. {image_path.name}")


# =============================================================================
# [5/9] Feature extraction function
# =============================================================================

print("\n[5/9] Preparing feature extraction...")
print("-" * 90)


def extract_features_from_result(result):
    """
    Convert YOLO detections for ONE image into the same
    21 features used when training the Random Forest.
    """

    # -------------------------------------------------------------------------
    # Image dimensions
    # -------------------------------------------------------------------------

    image = result.orig_img

    height, width = image.shape[:2]

    image_area = width * height


    # -------------------------------------------------------------------------
    # Get YOLO detections
    # -------------------------------------------------------------------------

    if result.boxes is None or len(result.boxes) == 0:

        # No people detected.
        #
        # Return zeros for all numerical features.

        return {
            "person_count": 0,
            "avg_confidence": 0,
            "min_confidence": 0,
            "max_confidence": 0,

            "total_bbox_area_ratio": 0,
            "avg_bbox_area_ratio": 0,
            "max_bbox_area_ratio": 0,

            "avg_bbox_width_ratio": 0,
            "avg_bbox_height_ratio": 0,

            "people_per_megapixel": 0,

            "average_center_distance": 0,
            "average_nearest_neighbor_distance": 0,

            "grid_00": 0,
            "grid_01": 0,
            "grid_02": 0,
            "grid_10": 0,
            "grid_11": 0,
            "grid_12": 0,
            "grid_20": 0,
            "grid_21": 0,
            "grid_22": 0
        }


    # -------------------------------------------------------------------------
    # Extract boxes and confidence scores
    # -------------------------------------------------------------------------

    boxes = result.boxes.xyxy.cpu().numpy()
    confidences = result.boxes.conf.cpu().numpy()


    # -------------------------------------------------------------------------
    # Number of people
    # -------------------------------------------------------------------------

    person_count = len(boxes)


    # -------------------------------------------------------------------------
    # Confidence statistics
    # -------------------------------------------------------------------------

    avg_confidence = float(np.mean(confidences))
    min_confidence = float(np.min(confidences))
    max_confidence = float(np.max(confidences))


    # -------------------------------------------------------------------------
    # Bounding-box statistics
    # -------------------------------------------------------------------------

    bbox_widths = boxes[:, 2] - boxes[:, 0]
    bbox_heights = boxes[:, 3] - boxes[:, 1]

    bbox_areas = bbox_widths * bbox_heights

    bbox_area_ratios = bbox_areas / image_area

    total_bbox_area_ratio = float(np.sum(bbox_area_ratios))
    avg_bbox_area_ratio = float(np.mean(bbox_area_ratios))
    max_bbox_area_ratio = float(np.max(bbox_area_ratios))

    avg_bbox_width_ratio = float(
        np.mean(bbox_widths / width)
    )

    avg_bbox_height_ratio = float(
        np.mean(bbox_heights / height)
    )


    # -------------------------------------------------------------------------
    # People per megapixel
    # -------------------------------------------------------------------------

    image_megapixels = image_area / 1_000_000

    people_per_megapixel = float(
        person_count / image_megapixels
    )


    # -------------------------------------------------------------------------
    # Bounding-box centers
    # -------------------------------------------------------------------------

    centers_x = (boxes[:, 0] + boxes[:, 2]) / 2
    centers_y = (boxes[:, 1] + boxes[:, 3]) / 2

    centers = np.column_stack([
        centers_x / width,
        centers_y / height
    ])


    # -------------------------------------------------------------------------
    # Average center distance
    # -------------------------------------------------------------------------

    if person_count >= 2:

        pairwise_distances = []

        for i in range(person_count):

            for j in range(i + 1, person_count):

                distance = np.linalg.norm(
                    centers[i] - centers[j]
                )

                pairwise_distances.append(distance)

        average_center_distance = float(
            np.mean(pairwise_distances)
        )

    else:

        average_center_distance = 0.0


    # -------------------------------------------------------------------------
    # Average nearest-neighbor distance
    # -------------------------------------------------------------------------

    if person_count >= 2:

        nearest_distances = []

        for i in range(person_count):

            distances = np.linalg.norm(
                centers - centers[i],
                axis=1
            )

            distances[i] = np.inf

            nearest_distances.append(
                np.min(distances)
            )

        average_nearest_neighbor_distance = float(
            np.mean(nearest_distances)
        )

    else:

        average_nearest_neighbor_distance = 0.0


    # -------------------------------------------------------------------------
    # 3 × 3 spatial grid
    # -------------------------------------------------------------------------

    grid_counts = np.zeros((3, 3), dtype=int)

    for cx, cy in centers:

        col = min(int(cx * 3), 2)
        row = min(int(cy * 3), 2)

        grid_counts[row, col] += 1


    # -------------------------------------------------------------------------
    # Return exactly the same 21 features used for training
    # -------------------------------------------------------------------------

    return {

        "person_count": person_count,

        "avg_confidence": avg_confidence,
        "min_confidence": min_confidence,
        "max_confidence": max_confidence,

        "total_bbox_area_ratio": total_bbox_area_ratio,
        "avg_bbox_area_ratio": avg_bbox_area_ratio,
        "max_bbox_area_ratio": max_bbox_area_ratio,

        "avg_bbox_width_ratio": avg_bbox_width_ratio,
        "avg_bbox_height_ratio": avg_bbox_height_ratio,

        "people_per_megapixel": people_per_megapixel,

        "average_center_distance": average_center_distance,
        "average_nearest_neighbor_distance":
            average_nearest_neighbor_distance,

        "grid_00": grid_counts[0, 0],
        "grid_01": grid_counts[0, 1],
        "grid_02": grid_counts[0, 2],

        "grid_10": grid_counts[1, 0],
        "grid_11": grid_counts[1, 1],
        "grid_12": grid_counts[1, 2],

        "grid_20": grid_counts[2, 0],
        "grid_21": grid_counts[2, 1],
        "grid_22": grid_counts[2, 2]
    }


# =============================================================================
# [6/9] Run complete pipeline
# =============================================================================

print("\n[6/9] Running complete YOLO → Random Forest pipeline...")
print("-" * 90)

results_table = []

pipeline_start = time.time()


for index, image_path in enumerate(selected_images, start=1):

    print(f"\n{'─' * 80}")
    print(f"IMAGE {index}/{len(selected_images)}")
    print(f"{'─' * 80}")

    print(f"Image: {image_path.name}")

    # -------------------------------------------------------------------------
    # YOLO inference
    # -------------------------------------------------------------------------

    start = time.time()

    yolo_result = yolo_model.predict(
        source=str(image_path),
        conf=CONF_THRESHOLD,
        iou=IOU_THRESHOLD,
        max_det=MAX_DETECTIONS,
        verbose=False
    )[0]

    inference_time = time.time() - start


    # -------------------------------------------------------------------------
    # Extract 21 features
    # -------------------------------------------------------------------------

    features = extract_features_from_result(
        yolo_result
    )


    # -------------------------------------------------------------------------
    # Create feature DataFrame
    # -------------------------------------------------------------------------

    feature_df = pd.DataFrame(
        [features]
    )


    # -------------------------------------------------------------------------
    # Random Forest prediction
    # -------------------------------------------------------------------------

    prediction = int(
        rf_model_v1.predict(feature_df)[0]
    )

    probability = float(
        rf_model_v1.predict_proba(feature_df)[0][1]
    )


    # -------------------------------------------------------------------------
    # Convert prediction to human-readable label
    # -------------------------------------------------------------------------

    if prediction == 1:

        final_label = "CROWD"

    else:

        final_label = "NOT-CROWD"


    # -------------------------------------------------------------------------
    # Store result
    # -------------------------------------------------------------------------

    results_table.append({

        "image": image_path.name,

        "person_count":
            features["person_count"],

        "avg_confidence":
            features["avg_confidence"],

        "people_per_megapixel":
            features["people_per_megapixel"],

        "nearest_neighbor_distance":
            features[
                "average_nearest_neighbor_distance"
            ],

        "crowd_probability":
            probability,

        "prediction":
            final_label,

        "yolo_time_sec":
            inference_time
    })


    # -------------------------------------------------------------------------
    # Print result
    # -------------------------------------------------------------------------

    print(
        f"Detected people       : "
        f"{features['person_count']}"
    )

    print(
        f"Average confidence    : "
        f"{features['avg_confidence']:.4f}"
    )

    print(
        f"People / megapixel    : "
        f"{features['people_per_megapixel']:.2f}"
    )

    print(
        f"Nearest-neighbor dist : "
        f"{features['average_nearest_neighbor_distance']:.4f}"
    )

    print(
        f"CROWD probability     : "
        f"{probability * 100:.2f}%"
    )

    print(
        f"FINAL PREDICTION      : "
        f"{final_label}"
    )

    print(
        f"YOLO inference time   : "
        f"{inference_time:.4f} sec"
    )


pipeline_time = time.time() - pipeline_start


# =============================================================================
# [7/9] Results table
# =============================================================================

print("\n[7/9] Complete pipeline results")
print("-" * 90)

results_df = pd.DataFrame(results_table)

display(
    results_df.style.format({
        "avg_confidence": "{:.3f}",
        "people_per_megapixel": "{:.2f}",
        "nearest_neighbor_distance": "{:.4f}",
        "crowd_probability": "{:.3f}",
        "yolo_time_sec": "{:.4f}"
    })
)

print(f"\nTotal pipeline time : {pipeline_time:.2f} seconds")


# =============================================================================
# [8/9] Visualize final predictions
# =============================================================================

print("\n[8/9] Visualizing YOLO detections + final predictions...")
print("-" * 90)


for index, image_path in enumerate(selected_images):

    # Run YOLO again for visualization.
    result = yolo_model.predict(
        source=str(image_path),
        conf=CONF_THRESHOLD,
        iou=IOU_THRESHOLD,
        max_det=MAX_DETECTIONS,
        verbose=False
    )[0]

    image = result.orig_img

    # Convert BGR → RGB for matplotlib.
    image_rgb = image[:, :, ::-1]

    # Retrieve corresponding prediction.
    row = results_df.iloc[index]

    prediction = row["prediction"]
    probability = row["crowd_probability"]
    person_count = int(row["person_count"])


    # -------------------------------------------------------------------------
    # Create figure
    # -------------------------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=(12, 8)
    )

    ax.imshow(image_rgb)

    ax.set_title(
        f"V1 Prediction: {prediction} | "
        f"People: {person_count} | "
        f"CROWD Probability: {probability * 100:.1f}%",
        fontsize=14
    )


    # -------------------------------------------------------------------------
    # Draw bounding boxes
    # -------------------------------------------------------------------------

    boxes = result.boxes.xyxy.cpu().numpy()

    for box in boxes:

        x1, y1, x2, y2 = box

        width = x2 - x1
        height = y2 - y1

        rectangle = patches.Rectangle(
            (x1, y1),
            width,
            height,
            linewidth=1.5,
            fill=False
        )

        ax.add_patch(rectangle)


    ax.axis("off")

    plt.tight_layout()
    plt.show()


# =============================================================================
# [9/9] Final V1 pipeline summary
# =============================================================================

print("\n[9/9] Final pipeline summary")
print("-" * 90)

crowd_count = sum(
    results_df["prediction"] == "CROWD"
)

not_crowd_count = sum(
    results_df["prediction"] == "NOT-CROWD"
)

print(f"""
Images tested          : {len(results_df)}

CROWD predictions      : {crowd_count}
NOT-CROWD predictions  : {not_crowd_count}

Average detected people:
    {results_df['person_count'].mean():.2f}

Average CROWD probability:
    {results_df['crowd_probability'].mean() * 100:.2f}%

Total pipeline time:
    {pipeline_time:.2f} seconds

PIPELINE:

    Input Image
        ↓
    YOLO26m
        ↓
    Person Detection
        ↓
    21 Scene Features
        ↓
    Random Forest
        ↓
    CROWD / NOT-CROWD
""")

print("=" * 90)
print(" CELL 11 COMPLETE — END-TO-END V1 PIPELINE VERIFIED")
print("=" * 90)

In [ ]:
# =============================================================================
# CELL 11 — CONTINUE AFTER DISPLAY ERROR
# =============================================================================
#
# The YOLO → Random Forest pipeline already completed successfully.
#
# The previous error happened because Cell 10 overwrote Python's
# notebook `display()` function with a ConfusionMatrixDisplay object.
#
# We avoid that problem by using print() instead of display().
# =============================================================================

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches

print("=" * 90)
print(" CELL 11 CONTINUATION — END-TO-END RESULTS")
print("=" * 90)


# =============================================================================
# [7/9] Create and display results table
# =============================================================================

print("\n[7/9] Complete pipeline results")
print("-" * 90)

results_df = pd.DataFrame(results_table)

print("\n")

# Print a clean table without using the overwritten display() function.
print(
    results_df[
        [
            "image",
            "person_count",
            "avg_confidence",
            "people_per_megapixel",
            "nearest_neighbor_distance",
            "crowd_probability",
            "prediction",
            "yolo_time_sec"
        ]
    ].to_string(index=False)
)

print(f"\nTotal pipeline time : {pipeline_time:.2f} seconds")


# =============================================================================
# [8/9] Visualize final predictions
# =============================================================================

print("\n[8/9] Visualizing YOLO detections + final predictions...")
print("-" * 90)

for index, image_path in enumerate(selected_images):

    # Run YOLO for visualization.
    result = yolo_model.predict(
        source=str(image_path),
        conf=CONF_THRESHOLD,
        iou=IOU_THRESHOLD,
        max_det=MAX_DETECTIONS,
        verbose=False
    )[0]

    image = result.orig_img

    # BGR → RGB
    image_rgb = image[:, :, ::-1]

    # Corresponding Random Forest prediction
    row = results_df.iloc[index]

    prediction = row["prediction"]
    probability = row["crowd_probability"]
    person_count = int(row["person_count"])

    # -------------------------------------------------------------------------
    # Create figure
    # -------------------------------------------------------------------------

    fig, ax = plt.subplots(figsize=(12, 8))

    ax.imshow(image_rgb)

    ax.set_title(
        f"V1 Prediction: {prediction} | "
        f"People: {person_count} | "
        f"CROWD Probability: {probability * 100:.1f}%",
        fontsize=14
    )

    # -------------------------------------------------------------------------
    # Draw YOLO bounding boxes
    # -------------------------------------------------------------------------

    boxes = result.boxes.xyxy.cpu().numpy()

    for box in boxes:

        x1, y1, x2, y2 = box

        box_width = x2 - x1
        box_height = y2 - y1

        rectangle = patches.Rectangle(
            (x1, y1),
            box_width,
            box_height,
            linewidth=1.5,
            fill=False
        )

        ax.add_patch(rectangle)

    ax.axis("off")

    plt.tight_layout()
    plt.show()


# =============================================================================
# [9/9] Final V1 pipeline summary
# =============================================================================

print("\n[9/9] Final V1 pipeline summary")
print("-" * 90)

crowd_count = int(
    (results_df["prediction"] == "CROWD").sum()
)

not_crowd_count = int(
    (results_df["prediction"] == "NOT-CROWD").sum()
)

average_people = results_df["person_count"].mean()

average_probability = results_df["crowd_probability"].mean()

average_inference = results_df["yolo_time_sec"].mean()


print(f"""
Images tested              : {len(results_df)}

CROWD predictions           : {crowd_count}
NOT-CROWD predictions       : {not_crowd_count}

Average detected people     : {average_people:.2f}

Average CROWD probability   : {average_probability * 100:.2f}%

Average YOLO inference time : {average_inference:.4f} sec/image


COMPLETE V1 PIPELINE:

    Input Image
         ↓
    YOLO26m
         ↓
    Person Detection
         ↓
    21 Scene Features
         ↓
    Random Forest
         ↓
    CROWD / NOT-CROWD
""")


print("=" * 90)
print(" CELL 11 COMPLETE — END-TO-END V1 PIPELINE VERIFIED")
print("=" * 90)